# <font color="#418FDE" size="6.5" uppercase>**B: StyleGAN Experiment**</font>
----

Last update: 20240504

By the end of this lecture, you will be able to:
* Develop [Style](https://arxiv.org/pdf/1812.04948.pdf)/[Pro](https://arxiv.org/pdf/1710.10196.pdf)Gan-like generative models in the TensorFlow ecosystem.
* Build Generative Adversarial Networks' (GANs) functions & classes.
 * Near production-level functions & classes include [type hints](https://docs.python.org/3/library/typing.html) & formal descriptions with the [PEP 8](https://peps.python.org/pep-0008/) style implementation; we will see examples of them in this notebook.
 * Some of the codes are inspired/adopted from [High-Resolution Generative Adversarial Networks (GANs)](https://www.udemy.com/course/high-resolution-generative-adversarial-networks).

## **1. StyleGAN, StyleGAN2, & ProGAN Overview**

[StyleGAN](https://arxiv.org/pdf/1812.04948.pdf), [StyleGAN2](https://arxiv.org/pdf/1912.04958.pdf), & [ProGAN](https://arxiv.org/pdf/1710.10196.pdf) are advanced neural network architectures developed by NVIDIA for generating highly realistic images. Each represents a significant step in the evolution of Generative Adversarial Networks (GANs).

> **[ProGAN](https://arxiv.org/pdf/1710.10196.pdf) (Progressive Growing of GANs)**
* Introduced in 2017, ProGAN is a technique that enhances the stability & quality of the training process for GANs.
* The key innovation of ProGAN is the progressive growing of both the generator & discriminator networks during training. This means starting with low-resolution images & gradually increasing the resolution by adding new layers to the networks.
* This approach helps the model to initially learn the large-scale structure of the image distribution & progressively learn finer details, resulting in high-quality, high-resolution images.
* ProGAN was a breakthrough in generating highly realistic images, particularly in domains like human faces.

> **[StyleGAN](https://arxiv.org/pdf/1812.04948.pdf)**
* Building on the successes of ProGAN, StyleGAN was introduced by NVIDIA in 2018. StyleGAN makes several improvements over ProGAN, most notably in the way it introduces a new style-based generator architecture.
* This architecture allows StyleGAN to control the style of the generated images at different levels of detail through a technique called "style mixing".
* The generator in StyleGAN uses a mapping network to transform a latent code into an intermediate latent space that governs different stylistic aspects of the output image, from coarse features (like pose & face shape) to finer details (like hair style & facial features).
* This approach not only improves image quality but also enhances the variability & disentanglement of the learned features.

> **[StyleGAN2](https://arxiv.org/pdf/1912.04958.pdf)**
* Released in 2019, StyleGAN2 addresses several artifacts & issues that were present in the first version of StyleGAN.
* These artifacts, which were often perceptible as unnatural features in generated images, were largely due to certain design choices in StyleGAN’s architecture & training methods.
* StyleGAN2 introduces refinements in the network architecture, particularly in the normalization methods & the way styles are injected, resulting in cleaner & more coherent images.
* This version also improved the training methodology, which enhanced the stability & quality of the generated images.

Each of these models represents a significant step forward in the field of AI, particularly in image generation, offering tools that can create increasingly realistic & high-resolution images from purely random noise. These models have wide applications, including in graphics design, entertainment, & even academic research where synthetic image generation is required.

In [ ]:
#@title All Required Libraries:
import os
import glob
import copy
import math
import random
import time
import shutil

import tensorflow as tf
import numpy as np

from tensorflow.python.keras.utils import conv_utils
from matplotlib import pyplot as plt
from typing import *


## **2. Proposed GAN Model Architecture**

### **2.1. Generator Architecture**

The generator process begins with a latent space vector, which contains the random noise that seeds the image generation. This vector is fed into the network, starting the generation process at a very low resolution of 4x4.

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/generator_experiment.png?raw=true" width="75%">
  <br>
  <figcaption>Figure: Generator architecture.
</div>

As the data flows through the network, the image is progressively upsampled & refined through a series of blocks, which increases the resolution at each step (from 4x4 to 8x8, then 16x16, & so on, up to 512x512). After each upsampling operation, there is an "Add" operation. This refers to a residual connection that adds the upscaled version of the lower-resolution image to the output of the next block. This technique helps to stabilize the training process by allowing gradients to flow through the network more effectively & also helps to incorporate details from the lower resolution steps into the higher resolution outputs.

Each resolution block likely transforms the image in ways that add finer details, with earlier blocks establishing the broad strokes of the image & later blocks refining the textures & colors. The use of such blocks is consistent with StyleGAN architectures, which use a concept called "style" to control the generation process at different levels of the network.

At the output end of the generator, there is a step that converts the last block's output into a 3-channel image, which is the standard format for a colored image (representing the red, green, & blue channels).

In this architecture, the upscaling of the image & the addition of details at each step are crucial. The network learns to craft an image starting from the most abstract representation (latent space) & gradually adding layers of complexity until a detailed & coherent image is produced. This strategy is quite powerful in generating high-resolution images that are diverse & often quite realistic, which is the hallmark of the StyleGAN family of generators.

### **2.2. Discriminator Architecture**



In our GAN model, the discriminator's architecture is designed to process images of varying resolutions, which could represent either generated (fake) or real images. The process starts with the input image & progressively reduces its resolution through several stages, as indicated by the series of blocks with descending resolutions (512x512, 256x256, 128x128, down to 4x4).

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/discriminator_experiment.png?raw=true" width="75%">
  <br>
  <figcaption>Figure: Discriminator architecture <figcaption>
</div>

At each stage, there are connections that lead to a "Residual" block. In the context of neural networks, residual blocks usually refer to components that allow the learning of identity functions, which can facilitate the training of very deep networks by addressing issues like vanishing gradients. This approach might be inspired by residual learning techniques seen in architectures like [ResNet](https://keras.io/api/applications/resnet/), & adapted here for GANs.

The discriminator outputs a "Realism Score," which is typical of GAN discriminators that assess whether the input image is real or fake. The score is continuous, representing the discriminator's confidence in its assessment, ranging from negative to positive infinity.

The discriminator architecture is hierarchical, which is characteristic of StyleGAN & StyleGAN2, where the discriminator evaluates images at progressively coarser scales, allowing it to focus on different levels of detail at each stage. This is an effective approach for high-resolution image synthesis & evaluation.

StyleGAN2’s architecture is known for its unique features like skip connections & adaptive normalization layers that contribute to its efficiency.

## **3. Parameter Scaling & Resampling**

### **3.1. Resampling**

In GANs, upsampling & downsampling are techniques used to modify the resolution of images during the training process. Here’s how each process works generally:

> **Downsampling**
* Downsampling reduces the resolution of an image.
* This process begins with a blurring step, which helps to merge details & reduce high-frequency noise in the image. This blurring is crucial as it prevents the loss of important features when the image size is reduced.
* After blurring, the image is subsampled, meaning only certain pixels are retained while others are discarded. Typically, every second pixel is kept.
* This results in a smaller image that retains the most significant aspects of the original but with fewer details.

> **Upsampling**
* Upsampling is the technique used to increase the resolution of an image. This involves enlarging the image & then interpolating new pixel values in a way that makes the image appear smoother & more detailed.
* In practice, new pixels are initially inserted as zero values between existing pixels (padded), effectively doubling the size of the image in terms of height & width.
* After this insertion, a blurring effect is applied over the entire image, which helps in blending these new pixels into the original image content.
* This blurring not only fills the gaps with meaningful data but also smoothens the overall image to reduce the appearance of jagged edges or pixelation.
* Since the resulting image after padding is darker (averaged each pixel with 4 zeros), each pixel is multiplied by 4.



Together, these techniques are essential in the training of GANs, where managing different image resolutions is crucial for the generator to learn how to create new, high-quality images, & for the discriminator to effectively categorize images.








In [ ]:
#@title Custom Function - Resampling
'''
Required libraries:
import tensorflow as tf
from typing import Any
'''

def create_blur_filter(dtype: Any) -> tf.Tensor:
    """
    Creates a Gaussian-like blur filter tensor.

    Args:
        dtype: The data type of the returned tensor (e.g., tf.float32).

    Returns:
        tf.Tensor: A 4x4 blur filter with specified dtype.
    """
    return tf.constant(
        [
            [0.015625, 0.046875, 0.046875, 0.015625],
            [0.046875, 0.140625, 0.140625, 0.046875],
            [0.046875, 0.140625, 0.140625, 0.046875],
            [0.015625, 0.046875, 0.046875, 0.015625]
        ],
        dtype=dtype
    )

def blur(x: tf.Tensor, strides: int = 1) -> tf.Tensor:
    """
    Applies a depthwise convolution to input tensor 'x' using a Gaussian-like blur filter.
    This function blurs the input image by applying a predefined kernel that approximates
    a Gaussian blur. The blurring is done via depthwise convolution, which applies
    a single filter to each input channel independently.

    Args:
        x (tf.Tensor): The input tensor to be blurred. Typically, this tensor should
                       have a shape of [batch_size, height, width, channels].
        strides (int): The stride size for the depthwise convolution operation. Strides
                       specify how much the filter moves in each step of the convolution.
                       A stride of 1 means the filter is moved one pixel at a time.

    Returns:
        tf.Tensor: The blurred output tensor, which has the same shape as the input tensor
                   but with the spatial information smoothed by the convolution.
    """

    # Determine the number of channels in the input tensor.
    # This is necessary to properly replicate the blur filter across all channels.
    channel_count = x.shape[3]

    # Create a 4x4 Gaussian-like blur filter specific to the data type of the input tensor.
    # This filter is initially 2D (height x width), & we need to expand its dimensions
    # to apply it across each channel independently.
    filter = create_blur_filter(x.dtype)[:, :, tf.newaxis, tf.newaxis]

    # Tile the filter across the channel dimension to ensure each channel receives the
    # same filter during the depthwise convolution. This expands the filter from 2D to 4D,
    # matching the number of dimensions required for depthwise convolution.
    filter = tf.tile(filter, [1, 1, channel_count, 1])

    # Apply the depthwise convolution using the replicated filter. The convolution applies
    # the filter to each input channel independently without mixing between channels.
    # The strides parameter controls how the filter is moved across the input tensor,
    # & 'SAME' padding ensures the output tensor has the same spatial dimensions as the input.
    return tf.nn.depthwise_conv2d(x, filter, strides=[1, strides, strides, 1], padding='SAME')

def downsample(x: tf.Tensor) -> tf.Tensor:
    """
    Downsamples the input tensor 'x' by a factor of 2 using a Gaussian-like blur filter and
    a stride of 2 in the convolution.

    Args:
        x (tf.Tensor): The input tensor to be downsampled.

    Returns:
        tf.Tensor: The downsampled output tensor.
    """
    return blur(x, strides=2)

def upsample(x: tf.Tensor) -> tf.Tensor:
    """
    Upsamples the input tensor 'x' by a factor of 2 using nearest neighbor interpolation
    followed by a blur to smooth the result. This method increases the spatial resolution
    of the image by inserting zeros between pixels & then applies a blur to integrate
    these new pixels into the original image context.

    Args:
        x (tf.Tensor): The input tensor to be upsampled. Expected to have the format
                       [batch_size, height, width, channels].

    Returns:
        tf.Tensor: The upsampled & blurred output tensor, which will have double the
                   height & width of the input tensor.
    """
    def upsample_with_zeros(x: tf.Tensor) -> tf.Tensor:
        """
        Internal function to perform nearest neighbor upsampling by inserting zeros
        between the pixels of the input tensor.

        Args:
            x (tf.Tensor): Input tensor with dimensions [batch_size, height, width, channels].

        Returns:
            tf.Tensor: Tensor with dimensions [batch_size, height*2, width*2, channels], where
                       zeros have been inserted between original pixels.
        """
        # Extract dimensions from the input tensor.
        in_height, in_width, channel_count = x.shape[1], x.shape[2], x.shape[3]

        # Calculate the output dimensions, which are double the input dimensions.
        out_height, out_width = in_height * 2, in_width * 2

        # Reshape the input tensor to prepare for zero insertion.
        # We create extra dimensions at height & width positions & then pad these
        # dimensions with zeros to interleave the original pixels with zeros.
        x = tf.reshape(x, [-1, in_height, 1, in_width, 1, channel_count])
        x = tf.pad(x, [[0, 0], [0, 0], [0, 1], [0, 0], [0, 1], [0, 0]])

        # Reshape back to the expected output dimensions with the new height & width.
        return tf.reshape(x, [-1, out_height, out_width, channel_count])

    # Apply the upsample_with_zeros function to the input tensor, multiplying by 4 to
    # compensate for the increase in area, so that the intensity of the image remains
    # consistent. Then, apply a blur to smooth the image, reducing artifacts introduced
    # by the zero-insertion upsampling method.
    return blur(upsample_with_zeros(x * 4.))


'''
Wrappers/Decorators:


Wrappers in programming serve several important roles, primarily enhancing
abstraction, modularity, & code reusability. By encapsulating complex
operations within a wrapper, developers can simplify their high-level code,
making it cleaner & easier to manage. Wrappers hide the underlying complexity
of tasks—such as data batching, augmentation, or specific configurations of
machine learning operations—allowing those details to be abstracted away from
the main application logic. This modularity facilitates independent development,
testing, & maintenance of each component, which is particularly beneficial in
large systems or when working within team environments.

Additionally, wrappers help in ensuring the integration & compatibility of
systems that may not naturally work well together, adapting data formats and
communication protocols as needed. They provide a uniform interface to
underlying functionalities, which can differ in their implementations across
various libraries or systems, enabling code to be more portable & adaptable
without extensive modifications. Wrappers also allow for the extension of
existing functionalities & finer control over component behavior, enhancing
customization for specific application needs without altering original source
codes. In contexts like TensorFlow, wrappers like Upsample & Downsample make
it feasible to integrate specific operations smoothly into neural network
architectures, promoting clean, manageable, & reusable code structures.

More: https://www.geeksforgeeks.org/function-wrappers-in-python/
More: https://www.youtube.com/watch?v=ZwMS3_Ej_6M
'''

class Upsample(tf.keras.layers.Layer):
    """
    A custom Keras layer that upsamples the input tensor using a predefined upsampling function.

    This layer acts as a wrapper that applies an upsampling operation defined outside of this class.
    It's intended to increase the spatial dimensions (height & width) of the input tensor.

    Methods:
        call(x): Performs the upsampling operation on the input tensor.
    """

    def call(self, x: tf.Tensor) -> tf.Tensor:
        """
        Call method for the upsampling layer.

        Args:
            x (tf.Tensor): Input tensor to the layer.

        Returns:
            tf.Tensor: The upsampled tensor.

        Notes:
            This method assumes that an `upsample` function is defined elsewhere in the codebase,
            which should handle the specifics of the upsampling operation (e.g., using nearest
            neighbor interpolation followed by a blur for smoothing).
        """
        return upsample(x)  # Assumes 'upsample' is defined elsewhere with the desired logic.

class Downsample(tf.keras.layers.Layer):
    """
    A custom Keras layer that downsamples the input tensor using a predefined downsampling function.

    This layer acts as a wrapper that applies a downsampling operation defined outside of this class.
    It's intended to reduce the spatial dimensions (height & width) of the input tensor.

    Methods:
        call(x): Performs the downsampling operation on the input tensor.
    """

    def call(self, x: tf.Tensor) -> tf.Tensor:
        """
        Call method for the downsampling layer.

        Args:
            x (tf.Tensor): Input tensor to the layer.

        Returns:
            tf.Tensor: The downsampled tensor.

        Notes:
            This method assumes that a `downsample` function is defined elsewhere in the codebase,
            which should handle the specifics of the downsampling operation (e.g., using a blur
            followed by a stride-based convolution to reduce dimensions).
        """
        return downsample(x)  # Assumes 'downsample' is defined elsewhere with the desired logic.


### **3.2. Parameter Scaling**

In GANs, scaling layers such as scaled convolutions & scaled activation functions like Leaky ReLU are to keep the weights mean & standard deviation close to 0 & 1, respectively, while maintaining the mean & standard deviation of the layer's output tensor around those of the layer's input tensor. They are used for a few key reasons:

* **Stabilizing Training:** GAN training can be highly unstable. This instability comes from the fact that GANs involve a min-max optimization process where two networks (the generator & the discriminator) are trained simultaneously with opposing goals. Scaling factors can help in maintaining the magnitude of the gradients during backpropagation, avoiding problems like exploding or vanishing gradients, which are common in deep networks.
* **Improved Gradient Flow:** Scaled activation functions, like scaled Leaky ReLU, ensure that the gradients have a controlled scale during the backward pass. This leads to a more stable & consistent flow of gradients across layers, essential for deep networks. This also allows the networks to learn efficiently from the data without saturating too quickly.
* **Weight Normalization:** Scaling layers can act as a form of weight normalization. By adjusting the scale of the weights, it's possible to maintain the activations within a range that is more suitable for the learning process. This normalization can lead to faster convergence during training & can improve the generalization of the model.
* **Equalized Learning Rate:** Specifically, in StyleGAN & its successors, scaled convolution layers are used as part of an "equalized learning rate" technique. The idea is to scale the weights dynamically at runtime to keep the learning rate the same across all layers of the network. This prevents early layers from learning too quickly & dominating the learning process, which can be a problem with GANs due to their depth & complexity.
* **Better Control Over Learning Dynamics:** By scaling layers & activation functions, developers have better control over the learning dynamics. It allows them to manipulate the learning process more delicately, which can lead to better generation of high-quality images or whichever data the GAN is being used to generate.

The use of scaled layers is an engineering solution that addresses some of the practical challenges in training deep generative models. It helps in achieving a balance between sufficient expressive power of the model & manageable computational complexity, leading to more stable training & better-performing models.

In [ ]:
#@title Scaled Layer - LeakyRelu
'''
Required libraries:
import math
import tensorflow as tf
from typing import Dict, Any
'''

class ScaledLeakyReLU(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies the Leaky ReLU activation function with a scaling factor.

    Leaky ReLU is an activation function defined as:
        f(x) = alpha * x for x < 0,
        f(x) = x for x >= 0,
    where `alpha` is a small coefficient. This layer further scales the output by a constant
    `gain` to control the magnitude of the output, which can be useful in maintaining
    the neural network's training dynamics.

    Attributes:
        alpha (float): Coefficient of leakage in the negative part of the function.
        gain (float): Scaling factor applied to the output of the activation function.
    """

    def __init__(self, alpha: float = 0.2, gain: float = math.sqrt(2.0), **kwargs):
        """
        Initialize the ScaledLeakyReLU layer.

        Args:
            alpha (float): Coefficient for the negative slope of the Leaky ReLU.
            gain (float): Gain factor to scale the output of the activation.
            **kwargs: Arbitrary keyword arguments for the base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.alpha = alpha
        self.gain = gain

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Logic for the layer's forward pass.

        Args:
            inputs (tf.Tensor): Input tensor.

        Returns:
            tf.Tensor: Output tensor after applying the Leaky ReLU activation & scaling.
        """
        return tf.nn.leaky_relu(inputs, self.alpha) * self.gain

    def get_config(self) -> Dict[str, Any]:
        """
        Returns the configuration of the layer as a dictionary for serialization.

        Returns:
            Dict[str, Any]: Configuration of the layer including `alpha` & `gain`.
        """
        config = super().get_config()
        config.update({
            'alpha': self.alpha,
            'gain': self.gain
        })
        return config


In [ ]:
#@title Scaled Layer - Conv2D
'''
Required libraries:
import tensorflow as tf
from typing import List, Dict, Any
'''

class ScaledConv2d(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies a scaled 2D convolution to the input (performs downsampling).

    This layer introduces a scaling factor to the convolution operation to manage the amplitude
    of outputs, potentially improving the stability & performance of neural networks.
    Optionally, a blur operation can be applied to the inputs before the convolution to reduce
    high-frequency noise.

    Attributes:
        channel_count (int): The number of output channels (filters) in the convolution.
        kernel_size (int): The dimensions of the convolution window (height & width).
        strides (int): The number of pixels by which the convolution window moves during the sliding.
        padding (str): The strategy for handling the border of the input ('valid' for no padding,
                       'same' to pad input so the output has the same width/height dimension).
        pre_blur (bool): A flag to determine whether to blur the input before the convolution.
    """

    def __init__(
        self,
        channel_count: int,
        kernel_size: int,
        strides: int = 1,
        padding: str = 'valid',
        pre_blur: bool = False,
        **kwargs
    ):
        """
        Initialize the ScaledConv2d layer with necessary configurations.

        Args:
            channel_count (int): Number of filters in the convolution.
            kernel_size (int): Size of the convolutional kernel (e.g., 3 for a 3x3 kernel).
            strides (int): Stride of the convolution.
            padding (str): Padding type, either 'valid' (no padding) or 'same' (pad to keep size).
            pre_blur (bool): Whether to pre-blur the inputs to soften features.
            **kwargs: Additional keyword arguments for the base Layer class.
        """
        super().__init__(**kwargs)
        # Normalizing parameters to ensure they are in acceptable formats.
        self.rank = 2  # Convolutional rank indicating it's a 2D convolution.
        self.channel_count = channel_count
        self.kernel_size = conv_utils.normalize_tuple(kernel_size, self.rank, 'kernel_size')
        self.strides = conv_utils.normalize_tuple(strides, self.rank, 'strides')
        self.padding = conv_utils.normalize_padding(padding)
        self.pre_blur = pre_blur

    def build(self, input_shape: List[int]) -> None:
        """
        Build the weights of the layer.

        Args:
            input_shape (List[int]): Shape of the input tensor, expected to include channel dimension.

        Raises:
            AssertionError: If the provided input shape does not meet the expected criteria.
        """
        # Checking to ensure the input shape is correct.
        assert len(input_shape) == self.rank + 2, "Input shape must match rank 2 plus batch & channel dimensions."

        # Configuring the shape of the weights based on input dimensions & the number of output channels.
        in_channel_count = input_shape[-1]
        kernel_shape = self.kernel_size + (in_channel_count, self.channel_count)
        self.kernel = self.add_weight(
            name='kernel',
            shape=kernel_shape,
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0),
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.channel_count,),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        # Scaling factor to adjust the amplitude of the output, thereby helping control the learning dynamics.
        self.scale = self.add_weight(
            name='scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(
                1.0 / tf.sqrt(tf.reduce_prod(tf.cast(kernel_shape[:-1], tf.float32)))),
            trainable=False
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Perform the convolution operation on input data.

        Args:
            inputs (tf.Tensor): Input tensor to be processed by convolution.

        Returns:
            tf.Tensor: The output tensor after applying the convolution & bias.
        """
        y = inputs
        if self.pre_blur:
            # Optionally apply a blur to the input to reduce noise & soften features.
            y = self.blur(y)  # Note: 'blur' should be a defined function or method in this or imported context.
        # Apply convolution with the scaled kernel & appropriate padding & stride settings.
        y = tf.nn.conv2d(
            y,
            self.kernel * self.scale,
            strides=self.strides,
            padding=self.padding.upper()
        )
        # Add the bias to the convoluted outputs.
        y = tf.nn.bias_add(y, self.bias)
        return y

    def get_config(self) -> Dict[str, Any]:
        """
        Serialize the configuration of the layer for storage or copying.

        Returns:
            Dict[str, Any]: A dictionary containing all configuration details of the layer.
        """
        config = super().get_config()
        config.update({
            'channel_count': self.channel_count,
            'kernel_size': self.kernel_size,
            'strides': self.strides,
            'padding': self.padding,
            'pre_blur': self.pre_blur
        })
        return config


In [ ]:
#@title Scaled Layer - UpsampleConv2d
'''
Required libraries:
import tensorflow as tf
from tensorflow.keras.utils import conv_utils
from typing import List, Dict, Any
'''

class UpsampleConv2d(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies a transposed convolution to upsample the input tensor.

    This layer increases the spatial dimensions of the input tensor using a transposed convolution,
    often referred to as a fractionally-strided convolution or a deconvolution. This operation is
    useful for tasks like image super-resolution, segmentation, & generative modeling where
    increasing image resolution or volume size is required.

    Attributes:
        channel_count (int): The number of output channels in the convolution.
    """

    def __init__(
        self,
        channel_count: int,
        **kwargs
    ):
        """
        Initializes the UpsampleConv2d layer with specified number of output channels.

        Args:
            channel_count (int): Number of output filters in the transposed convolution.
            **kwargs: Arbitrary keyword arguments for the base Layer class.
        """
        super().__init__(**kwargs)
        self.rank = 2  # Denotes 2D convolutional layers.
        self.channel_count = channel_count
        # Normalizing the kernel size to 3x3 & strides to 2 for upsampling.
        self.kernel_size = conv_utils.normalize_tuple(3, self.rank, 'kernel_size')
        self.strides = conv_utils.normalize_tuple(2, self.rank, 'strides')
        # Using 'same' padding to keep the output size as expected after convolution.
        self.padding = conv_utils.normalize_padding('same')

    def build(self, input_shape: List[int]) -> None:
        """
        Build the weights of the layer based on the input shape.

        Args:
            input_shape (List[int]): Shape of the input tensor.

        Raises:
            AssertionError: If the input shape does not have the expected dimensions.
        """
        assert len(input_shape) == self.rank + 2, "Input shape must be 4D (batch, height, width, channels)."
        in_channel_count = input_shape[-1]
        # Defining the shape of the kernel for the transposed convolution.
        kernel_shape = self.kernel_size + (self.channel_count, in_channel_count)
        self.kernel = self.add_weight(
            name='kernel',
            shape=kernel_shape,
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0),
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.channel_count,),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        # Scale to adjust the amplitude of the outputs.
        self.scale = self.add_weight(
            name='scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(
                1.0 / tf.sqrt(tf.cast(tf.reduce_prod(kernel_shape) // self.channel_count, tf.float32))),
            trainable=False
        )

    def compute_output_shape(self, input_shape):
        """
        Compute the output shape of the layer given the input shape.

        Args:
            input_shape: Shape of the input tensor.

        Returns:
            The expected output shape after the transposed convolution.
        """
        input_shape = tf.TensorShape(input_shape).as_list()
        # Output height & width are doubled due to the stride of 2.
        return tf.TensorShape([
            input_shape[0],
            input_shape[1] * 2,
            input_shape[2] * 2,
            self.channel_count])

    def call(self, x: tf.Tensor) -> tf.Tensor:
        """
        Perform the transposed convolution on the input tensor.

        Args:
            x (tf.Tensor): Input tensor.

        Returns:
            tf.Tensor: Upsampled tensor after applying transposed convolution & adding bias.
        """
        input_shape = tf.shape(x)
        batch_size, in_height, in_width = input_shape[0], input_shape[1], input_shape[2]
        output_shape = (batch_size, in_height * 2, in_width * 2, self.channel_count)

        # Perform the transposed convolution.
        y = tf.nn.conv2d_transpose(
            x,
            self.kernel * self.scale,
            output_shape,
            self.strides,
            padding=self.padding.upper()
        )

        # Ensure the shape is set during graph execution.
        if not tf.executing_eagerly():
            y.set_shape(self.compute_output_shape(x.shape))

        # Add the bias to the output.
        y = tf.nn.bias_add(y, self.bias)
        # Apply a blur for smoothing the output, assuming 'blur' is a defined function.
        y = blur(y) # removes the checkerboard artifact; see more at https://distill.pub/2016/deconv-checkerboard/
        return y

    def get_config(self) -> Dict[str, Any]:
        """
        Serialize the configuration of the layer to allow for model saving, loading, & cloning.

        Returns:
            A dictionary containing all configuration details of the layer.
        """
        config = super().get_config()
        config.update({
            'channel_count': self.channel_count
        })
        return config


In [ ]:
#@title Scaled Layer - Dense

'''
Required libraries:
import math
import tensorflow as tf
from typing import List, Dict, Any
'''

class ScaledDense(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies a scaled dense (fully connected) operation to the input.

    This layer creates a dense layer where the weights are scaled by a calculated factor based on
    the number of input units. This helps in maintaining a balanced variance across different sizes
    of input dimensions.

    Attributes:
        output_count (int): Number of neurons in the dense layer.
    """

    def __init__(
        self,
        output_count: int,
        **kwargs
    ):
        """
        Initialize the ScaledDense layer.

        Args:
            output_count (int): Number of output units (neurons) in the layer.
            **kwargs: Arbitrary keyword arguments for the base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.output_count = output_count

    def build(self, input_shape: List[int]) -> None:
        """
        Create the layer's weights.

        Args:
            input_shape (List[int]): Shape of the input tensor to the layer, should be 2D.

        Raises:
            AssertionError: If the input shape is not 2D.
        """
        assert len(input_shape) == 2, "Input shape must be 2D (batch_size, features)"
        self.kernel = self.add_weight(
            name='kernel',
            shape=(input_shape[-1], self.output_count),
            initializer=tf.keras.initializers.RandomNormal(mean=0.0, stddev=1.0),
            trainable=True
        )
        self.bias = self.add_weight(
            name='bias',
            shape=(self.output_count,),
            initializer=tf.keras.initializers.Zeros(),
            trainable=True
        )
        self.scale = self.add_weight(
            name='scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(
                1.0 / math.sqrt(input_shape[-1])),
            trainable=False
        )

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Logic for the layer's forward pass.

        Args:
            inputs (tf.Tensor): Input tensor.

        Returns:
            tf.Tensor: Output tensor after applying the scaled dense transformation & bias.
        """
        y = tf.matmul(inputs, self.kernel * self.scale)
        return tf.nn.bias_add(y, self.bias)

    def get_config(self) -> Dict[str, Any]:
        """
        Returns the configuration of the layer as a dictionary for serialization.

        Returns:
            Dict[str, Any]: Configuration of the layer.
        """
        config = super().get_config()
        config.update({
            'output_count': self.output_count
        })
        return config


In [ ]:
#@title Scaled Layer - Add
'''
Required libraries:
import math
import tensorflow as tf
from typing import List, Tuple, Dict, Any
'''

class ScaledAdd(tf.keras.layers.Layer):
    """
    A custom Keras layer that performs a scaled addition of two input tensors.

    This layer first adds two equally shaped tensors element-wise, & then scales
    the result by a constant factor. The scaling factor is a non-trainable weight
    initialized to a given value.

    Attributes:
        scale_value (float): The initial scaling factor applied to the summed inputs.
    """

    def __init__(self, scale: float = 1.0 / math.sqrt(2.0), **kwargs):
        """
        Initialize the ScaledAdd layer.

        Args:
            scale (float): The scaling factor for the addition operation. Default is
                           1/sqrt(2) to maintain a similar scale in transformations.
            **kwargs: Arbitrary keyword arguments for base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.scale_value = scale

    def build(self, input_shapes: Tuple[tf.TensorShape, tf.TensorShape]) -> None:
        """
        Create the layer's weights.

        Args:
            input_shapes (tuple of tf.TensorShape): Shapes of the two inputs. Both inputs
                                                   must have the same shape.

        Raises:
            AssertionError: If the input shapes do not match.
        """
        assert len(input_shapes) == 2, "ScaledAdd layer expects two inputs"
        a_shape, b_shape = input_shapes
        assert a_shape[1:] == b_shape[1:], f"Input shapes must match: {a_shape} != {b_shape}"

        self.scale = self.add_weight(
            name='scale',
            shape=(),
            initializer=tf.keras.initializers.Constant(value=self.scale_value),
            trainable=False
        )

    def call(self, inputs: List[tf.Tensor]) -> tf.Tensor:
        """
        The logic of the layer which performs the operations on the inputs.

        Args:
            inputs (list of tf.Tensor): A list containing two tensors to be added.

        Returns:
            tf.Tensor: The result of the scaled addition of the two input tensors.
        """
        return (inputs[0] + inputs[1]) * self.scale

    def get_config(self) -> Dict[str, Any]:
        """
        Returns the configuration of the layer as a dictionary.

        Returns:
            dict: A dictionary containing the configuration of the layer.
        """
        config = super().get_config()
        config.update({'scale': self.scale_value})
        return config


## **4. Minibatch Standard Deviation, Pixelwise Normalization, & Image Conversion**


### **4.1. Minibatch Standard Deviation**

In GANs, a common technique used to enhance the training & stability of the networks involves adding additional features derived from the data to the discriminator's input. One such feature augmentation method includes grouping real & fake images into batches, then computing the standard deviation of pixel values across these batches, & using these statistics as additional input features. Let's break down this process & explain why it's done:

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/minibatch_standard_deviation.png?raw=true" width="50%">
  <br>
  <figcaption>Figure: Implementing minibatch standard deviation as an additional feature to the feature map to reduce mode collapse in GANs.
</div>


> **Process Description**
* Grouping Images into Batches: Images (real & fake seperately) are grouped into batches. For real images, these could be images from the training dataset, while fake images are those generated by the generator part of the GAN.
* Computing Standard Deviation Across Batches: Within these batches, the standard deviation of the pixel values at corresponding positions (i.e., same pixel location across different images) is computed for every group of pixels. This calculation captures the variability of each pixel across different images within the batch.
* Averaging Standard Deviations: The standard deviations calculated in the previous step are then averaged. This average represents a typical variation for pixels in those positions across the batch of images.
* Spatial Replication of the Mean: The average standard deviation values are then spatially replicated to match the dimensions of the input images. This replication essentially creates a new channel or layer for the input image, where every pixel in this channel has a value corresponding to the averaged standard deviation of the corresponding pixel position in the original image.
* Adding as an Extra Feature: This new channel of averaged standard deviations is added to the original image as an additional feature. Now, each input to the discriminator has not only the original pixel values but also this extra channel depicting pixel variability across batches.

> **Reasons for Using This Technique**
* Improved Discriminator Sensitivity: By incorporating variability information (through standard deviations) directly into the input, the discriminator gains the ability to more effectively distinguish between real & fake images based on the consistency of pixel variations. Real images from natural sources typically exhibit certain consistent patterns of variability that might be hard for the generator to replicate accurately early in training.
* Regularization: Adding statistical features such as standard deviations can act as a form of regularization. This prevents the discriminator from overfitting to the exact pixel values of the training images & encourages it to focus on more general patterns that distinguish real images from generated ones.
* Enhanced Feature Representation: This approach enhances the feature space that the discriminator uses to make its decisions. It leverages not just the raw pixel values but also statistical properties derived from the dataset, making the discriminator's task both richer & potentially more complex, thus improving its robustness.
* Stability in Training: GAN training is notoriously unstable. Techniques that add extra information or modify the discriminator's input can help stabilize the training process by providing additional gradients & learning signals.
This technique showcases an innovative way to harness intra-batch variability to strengthen discriminator training, ultimately leading to the generation of more realistic synthetic images by the generator.

In [ ]:
#@title Custom Function - Minibatch Standard Deviation
'''
Required libraries:
import tensorflow as tf
from typing import Tuple, Optional, Any
'''

def reduce_std_nan_safe(x: tf.Tensor, axis: Optional[int] = None, keepdims: bool = False, epsilon: float = 1e-7) -> tf.Tensor:
    """
    Compute the standard deviation of a tensor along the specified axes, adding a small epsilon
    for numerical stability. This function is used to avoid division by zero when the variance
    is zero.

    Args:
        x (tf.Tensor): Input tensor whose standard deviation is to be calculated.
        axis (Optional[int]): The axis or axes along which to compute the mean & variance.
                              If None, standard deviation is computed over the whole tensor.
        keepdims (bool): If True, retains reduced dimensions with length 1 in the result.
        epsilon (float): A small constant added to the variance to improve numerical stability.

    Returns:
        tf.Tensor: A tensor containing the computed standard deviation, cast to the original
                   data type of input tensor `x`.

    Notes:
        The function first casts the input to float32 for stability in computations involving
        the mean & variance. It then calculates the mean of the tensor along the specified
        axis. The variance is determined as the average of the squared deviations from the mean.
        Adding a small epsilon before taking the square root ensures non-negative results, guarding
        against numerical issues. Finally, the result is cast back to the dtype of the input tensor
        to maintain consistency in data type.
    """
    # Cast input tensor to float32 to ensure stability in calculations
    y = tf.cast(x, tf.float32)

    # Compute the mean of the elements in the tensor along the specified axis
    mean = tf.reduce_mean(y, axis=axis, keepdims=True)

    # Calculate the variance as the mean of squared deviations from the mean
    variance = tf.reduce_mean(tf.square(y - mean), axis=axis, keepdims=keepdims)

    # Compute the standard deviation as the square root of variance, adding epsilon for numerical stability
    sqrt = tf.sqrt(variance + epsilon)

    # Cast the result back to the original data type of the input tensor
    return tf.cast(sqrt, x.dtype)

def minibatch_standard_deviation(x: tf.Tensor, group_size: int = 4) -> tf.Tensor:
    """
    Adds a minibatch standard deviation feature map to the last dimension of the input tensor.

    This function helps a neural network to take into account the variance of the minibatch, allowing
    it to learn from the internal statistics of the samples in each batch. It is often used in
    generative models like GANs to improve model stability & convergence.

    Args:
        x (tf.Tensor): Input tensor of shape (N, H, W, C) where:
            N = batch size
            H = height
            W = width
            C = channels
        group_size (int): The size of the groups into which the batch is split. Default is 4.

    Returns:
        tf.Tensor: Tensor with an additional feature map appended to the last channel, resulting in
                   the shape (N, H, W, C + 1).
    """
    # Get the original shape & data type of the input tensor
    original_shape = tf.shape(x)
    original_dtype = x.dtype

    # Compute the number of possible groups
    global_sample_count = original_shape[0]
    # Make sure we have at least three images; otherwise, the standard deviation is not working well.
    group_size = tf.minimum(group_size, global_sample_count)
    group_count = global_sample_count // group_size

    # Ensure the total number of elements is exactly divisible by the group size
    tf.Assert(
        group_size * group_count == global_sample_count,
        ['Sample count was not divisible by group size']
    )

    # Reshape the input to group the batch dimension into (group_count, group_size)
    y = tf.reshape(
        x,
        tf.concat([[group_count, group_size], original_shape[1:]], axis=0)
    )
    y = tf.cast(y, tf.float32)

    # Compute the standard deviation within each group
    stddevs = reduce_std_nan_safe(y, axis=1, keepdims=True)

    # Calculate the mean standard deviation across all groups & features
    avg_stddev = tf.reduce_mean(
        stddevs,
        axis=tf.range(1, tf.rank(stddevs)),
        keepdims=True
    )

    # Create a new feature map with the same spatial dimensions & append to the last channel
    new_feature_shape = tf.concat([tf.shape(y)[:-1], [1]], axis=0)
    new_feature = tf.broadcast_to(avg_stddev, new_feature_shape)
    y = tf.concat([y, new_feature], axis=-1)

    # Reshape back to the original batch dimension while preserving the new feature channel
    y = tf.reshape(
        y,
        tf.concat([[global_sample_count], original_shape[1:-1], [original_shape[-1] + 1]], axis=0)
    )
    y = tf.cast(y, original_dtype)

    return y

'''
Wrapper
'''

class MinibatchStandardDeviation(tf.keras.layers.Layer):
    """
    A custom Keras layer that adds a minibatch standard deviation feature map to the last dimension
    of the input tensor.

    This layer leverages the `minibatch_standard_deviation` function to add statistical variance
    features from the minibatch to the input tensor. This technique is commonly used in Generative
    Adversarial Networks (GANs) to enhance training stability & model convergence.

    Example:
        # Assuming `minibatch_standard_deviation` is defined elsewhere & imported.
        model = tf.keras.Sequential([
            tf.keras.layers.InputLayer(input_shape=(32, 32, 3)),
            MinibatchStandardDeviation()
        ])
    """

    def call(self, inputs: tf.Tensor) -> tf.Tensor:
        """
        Call method for the layer which gets invoked during model execution.

        Args:
            inputs (tf.Tensor): A 4D input tensor with shape (batch_size, height, width, channels).

        Returns:
            tf.Tensor: The input tensor with an added minibatch standard deviation feature map.
                       The resulting tensor shape is (batch_size, height, width, channels + 1).
        """
        # Call the externally defined function to add minibatch stddev feature to the input
        return minibatch_standard_deviation(inputs)


### **4.2. Pixelwise Normalization**

Pixelwise normalization is a technique introduced in [ProGAN](https://arxiv.org/pdf/1710.10196.pdf) to manage the escalation of signal magnitudes, a common challenge in the training of GANs.
* Unlike traditional normalization methods like batch normalization which normalize across a batch of data, pixelwise normalization operates at the level of individual pixels across the feature maps generated within the network.
* For each pixel in the width & height of a feature map, the method computes the square of each value across its channel, averages these squared values, & then takes the square root of this average.
* This result is used to normalize the original pixel value. The normalization is performed for each pixel separately, ensuring that the scale of the feature responses is uniform across the entire feature map.

<div align="left">
  <img src="https://github.com/mhrafiei/figures/blob/main/pixelwise_norm.png?raw=true" width="30%">
  <br>
  <figcaption>Figure: Pixelwise normalization: A tensor divided by the square root of the average of the square of all values across each channel.
</div>

The benefits of pixelwise normalization are significant, particularly in stabilizing the training dynamics of GANs.

* This method helps to control the scale of the activations & prevents the magnitudes of the generator model's output from escalating too quickly, which can lead to numerical instability & the collapsing of the model.
* By normalizing the output of each layer in such a localized manner, pixelwise normalization ensures that no single feature map dominates the learning process, promoting a more balanced & stable growth in the generator’s capabilities.
* Furthermore, this technique encourages the model to learn more meaningful & diverse features at different scales, contributing to the improved visual quality of the generated images.
* It effectively mitigates issues associated with unhealthy competition between the discriminator & generator, making it a crucial component in the architecture of more advanced GAN models.



In [ ]:
#@title Custom Function - Pixelwise Normalization
'''
Required libraries:
import tensorflow as tf
'''

def pixel_norm(x: tf.Tensor, epsilon: float = 1e-7) -> tf.Tensor:
    """
    Normalizes the input tensor by its pixel values to stabilize training in neural networks, particularly in GANs.

    Pixel normalization is performed per pixel over all channels. It divides each pixel by the square root of
    its mean squared value to normalize feature vectors to unit length, adding a small epsilon to the denominator
    for numerical stability to prevent division by zero.

    Args:
        x (tf.Tensor): The input tensor to normalize. Expected to have any shape with channels as the last dimension.
        epsilon (float): A small constant to ensure numerical stability by avoiding division by zero.

    Returns:
        tf.Tensor: The pixel-normalized tensor, cast back to the original data type of the input tensor.

    Notes:
        This function first casts the input tensor to float32 to ensure high precision during computation.
        The normalization uses the square root of the mean squared value across the channels, incorporating
        epsilon directly under the square root to maintain non-negative values. The result is then cast back
        to the original data type of the input tensor to maintain consistency in subsequent operations.
    """
    # Store the original data type of the input tensor to restore it after computation
    original_dtype = x.dtype

    # Cast the input tensor to float32 for high precision during division & square root operations
    x = tf.cast(x, tf.float32)

    # Compute the square of x, then average across the channels, add epsilon for numerical stability (in case of small vals)
    mean_sq = tf.reduce_mean(tf.square(x), axis=-1, keepdims=True) + epsilon

    # Normalize the input x by the square root of the mean squared value
    normalized = x / tf.math.sqrt(mean_sq)

    # Cast the normalized tensor back to the original data type of the input tensor
    return tf.cast(normalized, original_dtype)


'''
Wrapper
'''

class PixelNorm(tf.keras.layers.Layer):
    """
    A custom Keras layer that applies pixel normalization to an input tensor.

    Pixel normalization is a technique often used in generative adversarial networks (GANs) to
    stabilize the training process. It normalizes the feature vectors of the input tensor to
    unit length per pixel across the channels. This normalization helps manage scale discrepancies
    that might arise during training.

    Methods:
        call(x): Applies pixel normalization to the input tensor.
    """

    def call(self, x: tf.Tensor) -> tf.Tensor:
        """
        Apply the pixel normalization operation to the input tensor when the layer is called
        in a model.

        Args:
            x (tf.Tensor): Input tensor to be normalized.

        Returns:
            tf.Tensor: Pixel-normalized tensor. The normalization is performed per pixel across
                       the channels, scaling each feature vector to have a unit length.

        Notes:
            This method utilizes the 'pixel_norm' function. The function computes the normalization using the square root of
            the mean squared values of the tensor's channels, effectively scaling the pixel values
            to maintain unit length across channels.
        """
        return pixel_norm(x)  # Assumes 'pixel_norm' is defined with proper functionality.


### **4.3. Image Conversion**

In GANs, the decision to scale image pixel values between -1 & 1, rather than 0 & 1, is primarily driven by the benefits this scaling confers in terms of model training dynamics.

> **Training Stability & Performance**
* Using the range of -1 to 1 helps in stabilizing the training process of GANs. This scaling centers the data around zero & reduces the skewness which can lead to faster & more stable convergence in neural networks.
* The zero-centering aspect means that each feature (in this case, pixel intensity) has a mean of zero, leading to better symmetry & efficiency in gradient descent optimization.
* When data is scaled between 0 & 1, pixel values are only non-negative, which can shift the activation outputs & gradients during training, potentially leading to issues such as gradient vanishing or exploding.
* These problems are mitigated when the data is normalized around zero, allowing gradients to flow better during backpropagation & helping the network to learn more efficiently.

> **Symmetrical Data Distribution**
* Normalizing between -1 & 1 ensures a symmetrical distribution of input values, which is beneficial for the weights in the neural network.
* Weights can adjust in a more balanced manner without bias toward positive or negative value initialization, promoting a more uniform update during training across all neurons.
* This uniformity helps prevent certain weights from updating faster than others, which can lead to imbalanced learning & poor generation capabilities.

In [ ]:
#@title Custom Classes - Image Conversion
'''
Required libraries:
import tensorflow as tf
'''

class ImageConversion(tf.keras.layers.Layer):
    """
    A custom Keras layer for converting image data between different normalization ranges.

    This layer facilitates the conversion of image pixel values between two normalization
    schemes: [0, 1] & [-1, 1]. Depending on the `conversion_mode` specified during initialization,
    the layer can either normalize images (from [0, 1] to [-1, 1]) or denormalize them (from [-1, 1] to [0, 1]).

    Attributes:
        conversion_mode (int): Indicator of the conversion type to be applied. It supports:
            - -1 for [0, 1] to [-1, 1] conversion.
            - 0 for [-1, 1] to [0, 1] conversion.

    Methods:
        call(image): Applies the conversion to the image based on the `conversion_mode`.
        get_config(): Returns the configuration of the layer, including its `conversion_mode`.
    """

    def __init__(self, conversion_mode: int, **kwargs):
        """
        Initialize the ImageConversion layer with the specified mode of conversion.

        Args:
            conversion_mode (int): Mode of the conversion. -1 for normalizing to [-1, 1] and
                                    0 for denormalizing to [0, 1].
            **kwargs: Arbitrary keyword arguments for the base Layer class in Keras.
        """
        super().__init__(**kwargs)
        self.conversion_mode = conversion_mode

    def call(self, image: tf.Tensor) -> tf.Tensor:
        """
        Apply the specified image conversion when the layer is called.

        Args:
            image (tf.Tensor): Input tensor representing the image, with values expected to be in
                               the appropriate range based on the conversion mode.

        Returns:
            tf.Tensor: The converted image tensor.

        Raises:
            ValueError: If an unknown conversion mode is provided.
        """
        if self.conversion_mode == -1:
            return image * 2.0 - 1.0  # Convert from [0, 1] to [-1, 1]
        elif self.conversion_mode == 0:
            return image * 0.5 + 0.5  # Convert from [-1, 1] to [0, 1]
        else:
            # Instead of asserting, raise a ValueError for undefined conversion modes
            raise ValueError(f'Unknown conversion mode: {self.conversion_mode}')

    def get_config(self):
        """
        Overrides the default method to include the `conversion_mode` in the configuration.

        Returns:
            dict: Configuration dictionary containing the `conversion_mode` & base configuration.
        """
        config = super().get_config()
        config.update({
            'conversion_mode': self.conversion_mode
        })
        return config


## **5. Models, Loss, & Training Setup**

### **5.1. Generator & Discriminator Convolution Blocks**

In [ ]:
#@title Custom Function - Generator Convolution Block
'''
Required libraries:
import tensorflow as tf
'''

def generator_conv_block(filter_size: int, block1: tf.Tensor, block2: tf.Tensor, cond: bool = True) -> Tuple[tf.Tensor, tf.Tensor]:
    """
    Defines a convolutional block in the generator model.

    This function creates a convolutional block used in generator models within neural networks, particularly
    in generative models like GANs. It processes two input tensors through different pathways (often referred to as branches)
    & optionally applies an upsample operation based on a condition.

    Args:
        filter_size (int): Size of the convolution filters. This determines the number of output channels.
        block1 (tf.Tensor): Primary input tensor for the convolutional block. This tensor undergoes several convolutional transformations.
        block2 (tf.Tensor): Secondary input tensor, typically used for conditional operations such as feature concatenation.
        cond (bool): Flag to determine whether to apply an upsample operation to the output of the right branch.

    Returns:
        Tuple[tf.Tensor, tf.Tensor]: A tuple of tensors representing the output of the block; the modified `block1` & `block2`.
    """
    # Left branch of the convolution block:
    # 1. Upsampling followed by a 2D convolution increases the spatial dimensions & the number of channels based on `filter_size`.
    x0 = UpsampleConv2d(channel_count=filter_size)(block1)
    # 2. Apply a scaled leaky ReLU activation to add non-linearity to the output of the convolution.
    x1 = ScaledLeakyReLU()(x0)
    # 3. A second convolution is applied, maintaining the dimensions & filter size.
    x2 = ScaledConv2d(channel_count=filter_size, kernel_size=3, strides=1, padding='same')(x1)
    # 4. Another leaky ReLU activation is used after the second convolution.
    block1 = ScaledLeakyReLU()(x2)

    # Right branch of the convolution block:
    # 1. A convolution with a 1x1 kernel reduces the number of channels to 3 without changing spatial dimensions.
    x5 = ScaledConv2d(channel_count=3, kernel_size=1, strides=1, padding='same')(block1)
    # 2. Optionally add the second input tensor `block2` to the output of the right branch's convolution, if `block2` is not None.
    if block2 is not None:
        x5 = ScaledAdd()([x5, block2])

    # Conditional logic for post-processing:
    # 1. If `cond` is True, apply an upsample operation to the output of the right branch, increasing its spatial dimensions.
    if cond:
        block2 = Upsample()(x5)
    # 2. If `cond` is False, use the output from the right branch directly without upsampling.
    else:
        block2 = x5

    # The function returns the processed outputs of both branches as a tuple.
    return block1, block2


In [ ]:
#@title Custom Function - Discriminator Convolution Block
'''
Required libraries:
import tensorflow as tf
'''

def discriminator_conv_block(filter_size: tuple, block1: tf.Tensor) -> tf.Tensor:
    """
    Defines a convolutional block in the discriminator model.

    This function is responsible for constructing a convolutional block within the discriminator component of a GAN.
    The block processes an input tensor through parallel pathways (branches), applying convolutions, activations,
    & downsampling to transform the input tensor based on specified filter sizes.

    Args:
        filter_size (tuple): A tuple (filter_size_input, filter_size_output) specifying
                             the number of input & output channels for the convolutional layers.
        block1 (tf.Tensor): Input tensor for the convolutional block. This tensor is processed through
                            both the left & right branches of the block.

    Returns:
        tf.Tensor: Output tensor of the convolutional block, obtained by combining the results from both branches.
    """
    # Left branch of the block:
    # 1. Apply a convolution layer with the number of input channels specified in `filter_size[0]`.
    #    This layer uses a kernel size of 3 & maintains the spatial dimensions due to padding='same'.
    x0 = ScaledConv2d(channel_count=filter_size[0], kernel_size=3, strides=1, padding='same')(block1)
    # 2. Apply a leaky ReLU activation to introduce non-linearity to the output of the first convolution.
    x1 = ScaledLeakyReLU()(x0)
    # 3. Another convolution layer, now changing to the output channel size specified in `filter_size[1]` and
    #    reducing spatial dimensions by using a stride of 2.
    x2 = ScaledConv2d(channel_count=filter_size[1], kernel_size=3, strides=2, padding='same')(x1)
    # 4. Another application of a leaky ReLU activation function to add non-linearity to the processed tensor.
    x3 = ScaledLeakyReLU()(x2)

    # Right branch of the block:
    # 1. Downsample the input tensor to reduce its spatial dimensions. This typically halves the dimensions.
    x4 = Downsample()(block1)
    # 2. Apply a convolution layer to the downsampled tensor, using the output channel size `filter_size[1]`.
    #    This layer uses a kernel size of 3 & maintains spatial dimensions due to padding='same'.
    x5 = ScaledConv2d(channel_count=filter_size[1], kernel_size=3, strides=1, padding='same')(x4)

    # Combine the outputs of both branches by adding them together.
    # This element-wise addition helps to merge the features learned from both branches.
    return ScaledAdd()([x3, x5])


### **5.2. Loss Function**

In GANs, the choice of loss function plays a critical role in determining the convergence & stability of the training process. The paper by [Mescheder et al. (2018)](https://arxiv.org/pdf/1801.04406), titled "Which Training Methods for GANs do actually Converge?", examines several training methods & loss functions to understand their impact on the convergence behaviors of GANs. In this study, the authors recommend & prefer the R2 loss (also known as Least Squares GAN or LSGAN loss) over the Wasserstein loss under certain conditions, primarily due to its stability & convergence properties.

* The R2 loss is particularly favored because it theoretically & empirically leads to a more stable training regime compared to the original GAN loss formulations & even compared to Wasserstein loss in certain architectures.
* The R2 loss function minimizes the squared difference between the discriminator's outputs & the target values. This squared error loss penalizes incorrect classifications more smoothly & more significantly as the error increases, leading to a gradient behavior that is generally more favorable for stable training.
* The smoother gradient provided by the R2 loss helps prevent the discriminator from becoming too confident too quickly, a common issue in GAN training that can lead to the vanishing gradient problem for the generator.
* On the other hand, while the Wasserstein loss, as introduced by [Arjovsky et al. (2017)](https://arxiv.org/pdf/1701.07875), has been celebrated for improving the stability of GAN training by providing meaningful & smooth gradients even when the discriminator is very strong, it comes with its own set of challenges.
* One major issue is the requirement for the discriminator (critic) to lie within a set of 1-Lipschitz functions, which is typically enforced by weight clipping or gradient penalty.
* These constraints can complicate the training process & may not always guarantee better or even comparable convergence properties to the R2 loss.
* According to [Mescheder et al. (2018)](https://arxiv.org/pdf/1801.04406), the empirical findings suggest that the R2 loss provides more consistent & reliable convergence across different network architectures, making it a generally safer & more effective choice for training GANs.
* This preference for R2 loss is particularly highlighted in settings where the discriminator can significantly overpower the generator, helping maintain the delicate balance necessary for effective GAN training.
* The major component of our GAN loss in this experiment is the SoftPlus function.

<div align="middle">
  <img src="https://github.com/mhrafiei/figures/blob/main/softplus_plot.png?raw=true" width="30%">
  <br>
  <figcaption>Figure: Softplus function $f(x)=\log(1+\exp(x))$.
</div>


> **Discriminator Loss & Its Regularization**
* In GANs, the discriminator plays a crucial role in distinguishing between real & generated (fake) images. The loss function used for training the discriminator is designed to measure how well it performs this task. Specifically, the discriminator loss is calculated using a function called softplus, applied in two different ways: `softplus(fake_scores)` & `softplus(-real_scores)`.
* The `softplus(fake_scores)` component penalizes the discriminator when it fails to recognize fake images as fake. In other words, for each fake image that the discriminator incorrectly classifies as real, the softplus function generates a positive error value, contributing to the overall loss.
* Conversely, the `softplus(-real_scores)` part deals with real images; it penalizes the discriminator when it mistakenly classifies real images as fake. * The softplus function ensures that errors produce non-negative loss values, which are necessary for stable gradient calculations during training.
* Regularization is an essential technique used to prevent the discriminator from overfitting to the training data & to ensure that it generalizes well to new, unseen images.
* One common approach in GANs is to apply gradient penalty as a regularization technique. This involves periodically calculating the gradient of the discriminator's predictions with respect to the input images & then penalizing the model if these gradients grow too large, which can be indicative of overfitting.
* By constraining the magnitude of these gradients, the model is encouraged to learn smoother, more stable functions that are less sensitive to small fluctuations in input data.
* In this experiment, such regularization is computationally expensive; as such, it is only applied to the discriminator at certain predefined intervals (known as lazy regularization).

> **Generator Loss**
* The generator's goal in a GAN setup is to produce images that are indistinguishable from real images, effectively "fooling" the discriminator.
* The generator loss is formulated to reflect how well the generator is achieving this objective. Specifically, the loss is calculated using `softplus(-fake_scores)`, where `fake_scores` are the scores assigned by the discriminator to the images generated by the generator.
* The use of the softplus function here implies that the generator is penalized when the discriminator can easily distinguish its outputs as fake.
* If the discriminator classifies a generated image as clearly fake, resulting in a high score, the softplus of the negative of this score will result in a high loss value for the generator.
* Consequently, the generator is incentivized to adjust its parameters to produce images that will receive lower scores from the discriminator, indicating that they are more realistic.
* This loss mechanism drives the generator to improve continuously, enhancing the quality of the images it produces over time.

Both components of the GAN work in tandem through these loss functions, constantly learning from each other's responses & adjustments. The discriminator learns to better identify nuances that distinguish real from fake images, while the generator strives to create images that are increasingly difficult for the discriminator to classify correctly. This dynamic interplay is the core of what makes GANs powerful tools for generating high-quality synthetic images.


### **5.3. Experiment Arguments & Initiation**

> When preparing for a GAN training session, setting up the correct configuration & environment is crucial. This setup involves specifying various parameters that control the training process, model architecture, & management of data & output. The configuration for a GAN typically includes details such as the training data directory, the size & number of batches, the number of epochs for training, learning rates, & specific settings for both the generator & discriminator components of the network.

> The configuration starts by establishing a clear experiment name, which helps in organizing outputs & logs systematically. Details such as the directory for training data ensure the model knows where to fetch input images. Training parameters like batch size & number of epochs dictate the training duration & granularity. Moreover, specifying buffer sizes for data shuffling can significantly impact the randomness & quality of data fed into the model, which is vital for training robustness & avoiding overfitting.

> Additionally, the configuration includes operational settings like intervals for saving checkpoints & plotting generator outputs, which are essential for monitoring training progress & ensuring that you can resume training from a certain point if interrupted. Image size settings align the generator's output with the desired resolution, & the removal of existing data ensures that each training session starts fresh, avoiding contamination with outdated or irrelevant data files. Advanced settings include whether to run the model in inference mode to generate images without further training & selecting the computational resource (CPU or GPU), which could be crucial for performance optimization.

> To facilitate these extensive configurations, utility functions are often used to initialize & prepare the environment. These functions handle tasks like setting up directories for saving outputs like samples, checkpoints, & inferences, ensuring they exist before training begins. They also manage dynamic settings for model parameters based on specific requirements, such as adjusting filter sizes in the generator & discriminator based on the target image size, which directly influences the model's capacity & performance. By automating these setup steps through a function, GAN training becomes more manageable, reproducible, & adaptable to various experimental conditions.

> For our experiment, we will be utilizing the animal faces dataset, which initially comprises of over 15000 512 512-pixel PNG images. In order to reduce computation costs, we have generated a smaller version of this dataset, consisting of 32 by 32-pixel images. We will use this smaller version to train our GANs & observe how they learn to generate images of the same size. However, if you wish to use larger versions of these images, you will have to invest in superior GPUs & be prepared for a much longer training time. Here are the links to download the zip files of the larger versions of this dataset, which can be used with the Command Line Interface (CLI):
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_32.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_64.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_128.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_256.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_512.zip
* https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_1024.zip



In [ ]:
#@title Class - Args for training
'''
No required library.
'''

class Args():
    def __init__(self):
        """
        Initializes the configuration settings for a GAN training session.

        This class sets up various configuration parameters that control the training process, model architecture,
        & operational settings such as data handling & output management. These settings are crucial for customizing
        the behavior of the GAN according to specific experimental needs.
        """

        '''
        General Mandatory
        '''

        # Name of the experiment, used for organizing outputs & logs.
        self.experiment_name = 'animal_faces'
        # Type of the project
        self.project_type = 'gan'
        # The project patent folder
        self.project_parent_folder = '/content/drive/MyDrive'

        '''
        General Optional
        '''

        # The number of samples per batch to be used during training.
        self.batch_size = 64
        # Total number of epochs to train the models.
        self.epochs = 500
        # # Number of batches to process during the training. This can control how long an "epoch" is if dataset is very large.
        # self.batch_num = 60
        # Buffer size for shuffling the dataset. Larger sizes can improve shuffling quality but use more memory.
        self.buffer_size = 1000
        # Interval in batches to plot the generator output during training, helps monitor progress visually.
        self.plot_intervals = 50
        # Interval in batches to save checkpoints of the model weights.
        self.checkpoint_intervals = 50
        # Target size of images (height & width) that the generator should output.
        self.image_size = 128 # supported sizes are 2^n where n = [5,6,...,13]
        # If true, removes existing data from the folders like checkpoints & samples before starting new training.
        self.folder_remove = False
        # If true, runs the model in inference mode to generate images without training.
        self.inference_mode = False
        # Number of images to generate if running in inference mode.
        self.inference_num = 1
        # If true, forces the training or inference to run on CPU instead of GPU.
        self.gpu = True


        '''
        Generator
        '''

        # Size of the latent vector used as input to the generator.
        self.generator_latent_size = 512
        # Factor by which to reduce the filter size of each Conv layer from a baseline setting, affecting model capacity (2^n).
        self.generator_reduce_factor = 2 # acceptable values: 1, 2, 4, 8
        # Learning rate for the generator's optimizer.
        self.generator_lr = 0.002
        # First moment decay rate (beta1) for the Adam optimizer in the generator.
        self.generator_beta1 = 0.0
        # Second moment decay rate (beta2) for the Adam optimizer in the generator.
        self.generator_beta2 = 0.99

        '''
        Discriminator
        '''

        # Factor by which to reduce the filter size of each Conv layer in the discriminator, similar to generator's setting (2^n).
        self.discriminator_reduce_factor = 2 # acceptable values: 1, 2, 4, 8
        # Learning rate for the discriminator's optimizer.
        self.discriminator_lr = 0.002
        # First moment decay rate (beta1) for the Adam optimizer in the discriminator.
        self.discriminator_beta1 = 0.0
        # Second moment decay rate (beta2) for the Adam optimizer in the discriminator.
        self.discriminator_beta2 = 0.99
        # Interval at which to apply regularization techniques during discriminator training.
        self.discriminator_regularization_interval = 16


In [ ]:
#@title Counting Data in tfrecords
'''
Required Libraries:
import tensorflow as tf
'''

def count_records_in_tfrecord(tfrecord_path: str) -> int:
    """
    Count the number of records in a single TFRecord file.

    This function iterates through each record in the given TFRecord file & increments
    a counter to determine the total number of records.

    Args:
        tfrecord_path (str): The file path or URI to the TFRecord file. This can be a local
                             path or a remote path in a storage service like Google Cloud Storage
                             (indicated by a URI like gs://bucket_name/path_to_file).

    Returns:
        int: The total number of records in the specified TFRecord file.
    """
    count = 0
    # Initialize a TFRecordDataset to read from the specified file
    for _ in tf.data.TFRecordDataset(tfrecord_path):
        count += 1
    return count

def count_images_in_directory(directory_path: str) -> int:
    """
    Count the total number of images across all TFRecord files in a specified directory.

    This function first retrieves a list of all TFRecord files in the given directory,
    including support for remote directories in cloud storage. It then iterates through
    each TFRecord file, counting the records using `count_records_in_tfrecord`, and
    sums these counts to get the total number of images.

    Args:
        directory_path (str): The directory path containing TFRecord files. This can be a local
                              directory path or a remote directory path in a storage service
                              like Google Cloud Storage (e.g., "gs://bucket_name/path_to_directory/").

    Returns:
        int: The total number of records (images) across all TFRecord files in the directory.
    """
    total_count = 0
    # Use tf.io.gfile.glob to support both local & remote filesystems (e.g., GCS)
    tfrecord_files = tf.io.gfile.glob(os.path.join(directory_path, '*.tfrecord') )
    # Iterate through each file & count the records within
    for tfrecord_file in tfrecord_files:
        file_count = count_records_in_tfrecord(tfrecord_file)
        total_count += file_count
    return total_count


In [ ]:
#@title Functions - Initiation
'''
Required libraries:
import os
import shutil
import glob
import tensorflow as tf
from typing import *
'''

def initiate(args: Args) -> Dict[str, Any]:
    """
    Initializes the configuration for a GAN training session based on provided arguments.

    Args:
        args (Args): An instance of the Args class containing all necessary configuration parameters.

    Returns:
        Dict[str, Any]: A dictionary containing structured configuration settings for both the generator & discriminator,
                        as well as other operational settings necessary for the training or inference of the GAN.
    """

    # Initialize an empty dictionary to store configurations.
    opt                  = {}
    # Initialize sub-dictionaries for generator & discriminator configurations.
    opt['generator']     = {}
    opt['discriminator'] = {}
    # Set the number of inference samples based on the provided arguments.
    opt['inference_num'] = args.inference_num

    # Set the latent size for the generator based on arguments.
    opt['generator']['latent_size']     = args.generator_latent_size

    # Set filter sizes for both generator & discriminator based on the image size.
    # The filter sizes decrease as layers progress for the generator, & are tuples for discriminator adjustments.
    if args.image_size == 8192:
        opt['generator']['filter_size']     = [512,512,512,512,256,128,64,32,16,8,4]
        opt['discriminator']['filter_size'] = [(32, 64), (64, 128), (128, 256), (256, 512), (512, 512), (512, 512), (512, 512),(512, 512),(512, 512),(512, 512),(512, 512),(512, 512)]
    elif args.image_size == 4096:
        opt['generator']['filter_size']     = [512,512,512,512,256,128,64,32,16,8]
        opt['discriminator']['filter_size'] = [(32, 64), (64, 128), (128, 256), (256, 512), (512, 512), (512, 512), (512, 512),(512, 512),(512, 512),(512, 512),(512, 512)]
    elif args.image_size == 2048:
        opt['generator']['filter_size']     = [512,512,512,512,256,128,64,32,16]
        opt['discriminator']['filter_size'] = [(32, 64), (64, 128), (128, 256), (256, 512), (512, 512), (512, 512), (512, 512),(512, 512),(512, 512),(512, 512)]
    elif args.image_size == 1024:
        opt['generator']['filter_size']     = [512,512,512,512,256,128,64,32]
        opt['discriminator']['filter_size'] = [(32, 64), (64, 128), (128, 256), (256, 512), (512, 512), (512, 512), (512, 512),(512, 512),(512, 512)]
    elif args.image_size == 512:
        opt['generator']['filter_size']     = [512,512,512,512,256,128,64]
        opt['discriminator']['filter_size'] = [(64, 128), (128, 256), (256, 512), (512, 512), (512, 512), (512, 512),(512, 512),(512, 512)]
    elif args.image_size == 256:
        opt['generator']['filter_size']     = [512,512,512,512,256,128]
        opt['discriminator']['filter_size'] = [(128, 256), (256, 512), (512, 512), (512, 512), (512, 512),(512, 512),(512, 512)]
    elif args.image_size == 128:
        opt['generator']['filter_size']     = [512,512,512,512,256]
        opt['discriminator']['filter_size'] = [(256, 512), (512, 512), (512, 512), (512, 512),(512, 512),(512, 512)]
    elif args.image_size == 64:
        opt['generator']['filter_size']     = [512,512,512,512]
        opt['discriminator']['filter_size'] = [(512, 512), (512, 512), (512, 512),(512, 512),(512, 512)]
    elif args.image_size == 32:
        opt['generator']['filter_size']     = [512,512, 512]
        opt['discriminator']['filter_size'] = [(512, 512), (512, 512),(512, 512),(512, 512)]
    else:
        assert True, "Image_size is not supported"

    # Get reduced factors
    opt['generator']['reduce_factor'] = args.generator_reduce_factor
    opt['discriminator']['reduce_factor'] = args.discriminator_reduce_factor

    # Adjust filter sizes for both generator & discriminator based on reduction factors.
    opt['generator']['filter_size']     = [int(f/args.generator_reduce_factor) for f in opt['generator']['filter_size']]
    opt['discriminator']['filter_size'] = [( int(f1/args.discriminator_reduce_factor), int(f2/args.discriminator_reduce_factor) ) for f1, f2 in opt['discriminator']['filter_size']]

    # Set learning rate, beta1, & beta2 parameters for the generator based on provided arguments.
    opt['generator']['lr']              = args.generator_lr
    opt['generator']['beta1']           = args.generator_beta1
    opt['generator']['beta2']           = args.generator_beta2

    # Set the regularization interval for the discriminator.
    opt['discriminator']['regularization_interval'] = args.discriminator_regularization_interval
    # Calculate & set the lazy ratio for the discriminator regularization.
    opt['discriminator']['lazy_ratio']              = opt['discriminator']['regularization_interval'] / (opt['discriminator']['regularization_interval'] + 1)
    # Set learning rate, beta1, & beta2 parameters for the discriminator.
    opt['discriminator']['lr']                      = args.discriminator_lr
    opt['discriminator']['beta1']                   = args.discriminator_beta1
    opt['discriminator']['beta2']                   = args.discriminator_beta2
    # Set batch size, epochs, number of batches, buffer size, & intervals for plotting & checkpointing.
    opt['batch_size']           = args.batch_size
    opt['epochs']               = args.epochs
    opt['buffer_size']          = args.buffer_size
    opt['plot_intervals']       = args.plot_intervals
    opt['checkpoint_intervals'] = args.checkpoint_intervals
    # Set paths to the experiment-related folders.
    # experiment path which depends on the image_size.
    opt['dir_experiment'] = os.path.join(args.project_parent_folder, args.experiment_name, str(args.image_size), args.project_type)
    # Directory where the training data is located. This should be the path to a folder containing the training records.
    opt['dir_tfrecords'] = os.path.join(args.project_parent_folder, args.experiment_name, str(args.image_size), 'tfrecords')
    # Checkpoints path of discriminator & generators.
    opt['dir_checkpoints'] = os.path.join(opt['dir_experiment'], 'checkpoints')
    # Path to where sample-generated images are saved during training.
    opt['dir_samples'] = os.path.join(opt['dir_experiment'], 'samples')
    # Path to where inference-generated images are saved after training, during inference.
    opt['dir_inferences'] = os.path.join(opt['dir_experiment'], 'inferences')

    # Condition to check if existing training data directories should be removed.
    if args.folder_remove:
        if os.path.exists(opt['dir_experiment']):
            shutil.rmtree(opt['dir_experiment'])
        if os.path.exists(opt['dir_tfrecords']):
            shutil.rmtree(opt['dir_tfrecords'])
        if os.path.exists(opt['dir_checkpoints']):
            shutil.rmtree(opt['dir_checkpoints'])
        if os.path.exists(opt['dir_samples']):
            shutil.rmtree(opt['dir_samples'])
        if os.path.exists(opt['dir_inferences']):
            shutil.rmtree(opt['dir_inferences'])

    # Number of batches to process during the training.
    # This can control how long an "epoch" is if dataset is very large.
    total_images = count_images_in_directory(opt['dir_tfrecords'])
    opt['batch_num'] = int(total_images/args.batch_size)

    # Ensure the directories for samples, checkpoints, & inferences exist.
    os.makedirs(opt['dir_experiment'], exist_ok=True)
    os.makedirs(opt['dir_tfrecords'], exist_ok=True)
    os.makedirs(opt['dir_samples'], exist_ok=True)
    os.makedirs(opt['dir_checkpoints'], exist_ok=True)
    os.makedirs(opt['dir_inferences'], exist_ok=True)
    # Return the populated configuration dictionary.
    return opt


In [ ]:
#@title Class - MyGenerativeAdversarialModel
'''
Required libraries:
import os
import glob
import copy
import math
import random
import time
import shutil

import tensorflow as tf
import numpy as np

from tqdm import tqdm
from tensorflow.python.keras.utils import conv_utils
from matplotlib import pyplot as plt
from typing import *
'''

class MyGenerativeAdversarialModel:
    """
    A class for constructing & training a generative adversarial network (GAN) model.
    This class encapsulates both the generator & discriminator components, handling training,
    inference, & utility functions like saving & loading models.

    Attributes:
        opt (dict): Configuration options for various aspects of the GAN.
    """

    def __init__(self, opt: dict):
        """
        Initializes the GAN model with the specified options.

        This constructor method sets up a GAN by initializing its generator & discriminator components using configuration settings
        provided in a dictionary. It stores these settings & constructs the models according to specified parameters,
        making them ready for training or inference.

        Args:
            opt (dict): Configuration dictionary specifying generator & discriminator settings.
                        This dictionary must contain all necessary details such as the size of the latent space,
                        filter sizes, & any other model-specific parameters required for constructing the generator and
                        discriminator models.
        """
        # Store the configuration options in an instance variable for later use throughout the class.
        # This includes settings for both the generator & discriminator which define aspects like architecture depth,
        # learning rates, batch sizes, etc.
        self.opt = opt

        # Initialize the generator model:
        # The generator is created using a method that interprets the settings in `opt` to build the model's layers & configuration.
        # This part of the GAN generates data (e.g., images) from noise.
        self.generator = self.create_generator()

        # Initialize the discriminator model:
        # Similarly, the discriminator is constructed using another method that also uses settings from `opt`.
        # The discriminator's role is to evaluate whether given data is real or generated by the generator.
        self.discriminator = self.create_discriminator()


        # If checkpoints are available, load the weights on the generator & discriminator
        self.model_load()

    def create_generator(self) -> tf.keras.Model:
        """
        Constructs the generator model of the GAN using layers defined in
        configuration.

        This method constructs a generator model for a GAN, designed to produce
        outputs (like images) from noise inputs. The model is
        built progressively using multiple convolutional blocks, starting from a
        dense layer that reshapes the initial input into a format suitable for
        subsequent convolutional layers.

        Returns:
            tf.keras.Model: The constructed generator model. This model takes a
            latent vector as input & produces an output (e.g., an image).
        """
        # Define the input layer, which expects a latent vector of a specified size. The size is fetched from the configuration.
        inputs = tf.keras.layers.Input(shape=(self.opt['generator']['latent_size'],))

        # Reduce the Early Dense & Conv layer's units & filters based on the user-defined reduced factor
        dense_units = int(min(8192/self.opt['generator']['reduce_factor'], 8192))
        filter_size  = int(dense_units/16)

        # Initial block:
        # 1. A densely connected layer that maps the latent space to a higher dimension. The number dense_units represents the total neurons.
        block1 = ScaledDense(dense_units)(inputs)
        # 2. Reshape the output from the dense layer into a 4x4xfilter_size structure, preparing it for convolutional processing.
        #    This is a common practice in GANs to start with a small spatial footprint that increases in subsequent layers.
        block1 = tf.keras.layers.Reshape((4, 4, filter_size))(block1)
        # 3. Apply a leaky ReLU activation to introduce non-linearity, helping the model to learn more complex patterns.
        block1 = ScaledLeakyReLU()(block1)
        # 4. A convolutional layer with filter_size filters of size 3x3, used to further process the feature map without altering its size.
        block1 = ScaledConv2d(channel_count=filter_size, kernel_size=3, strides=1, padding='same')(block1)
        # 5. Another leaky ReLU activation follows the convolution to maintain the model's ability to learn non-linear features.
        block1 = ScaledLeakyReLU()(block1)

        # Construct further blocks dynamically:
        # Initialize `block2` as None, it will be used in the loop for operations that might require it.
        block2 = None
        for i, f in enumerate(self.opt['generator']['filter_size']):
            # Check if the current block is the last one. This information will dictate whether to apply certain operations like upsampling.
            is_last = i == len(self.opt['generator']['filter_size']) - 1
            # Invoke `generator_conv_block` to build & connect each block sequentially.
            # `cond` is passed as not `is_last` to ensure upsampling occurs only in intermediate blocks.
            block1, block2 = generator_conv_block(f, block1, block2, cond=not is_last)

        # After the loop, apply a final transformation to convert the tensor `block2` into an image-like format.
        outputs = ImageConversion(conversion_mode=0)(block2)
        # Construct & return the complete Keras model. It maps inputs (latent vectors) to outputs (images).
        return tf.keras.Model(inputs=inputs, outputs=outputs)

    def create_discriminator(self) -> tf.keras.Model:
        """
        Constructs the discriminator model of the GAN using layers defined in configuration.

        The discriminator is a critical component of a GAN that learns to distinguish between real & generated data.
        This method builds the discriminator model by stacking convolutional layers & applying various transformations,
        based on a predefined configuration. It starts by processing the input image & incrementally applying more complex
        filters & operations to capture features critical for classification.

        Returns:
            tf.keras.Model: The constructed discriminator model, which outputs a single value representing the model's
                            confidence in the authenticity of the input.
        """
        # Define the input layer to match the output shape of the generator's last layer,
        # which is dynamically fetched from the generator model. This ensures the discriminator
        # can process the exact output format produced by the generator.
        inputs = tf.keras.layers.Input(shape=tuple(self.generator.layers[-1].output_shape[1:]))

        # Initial block:
        # Convert the input image from the generator's output format to a standard format expected by the discriminator.
        block1 = ImageConversion(conversion_mode=-1)(inputs)
        # Apply a convolutional layer to the converted image, initializing the process of feature extraction.
        # The number of filters & other parameters are fetched from the configuration.
        block1 = ScaledConv2d(channel_count=self.opt['discriminator']['filter_size'][0][0],
                            kernel_size=3, strides=1, padding='same')(block1)
        # Introduce a leaky ReLU activation to allow the model to learn non-linear complexities.
        block1 = ScaledLeakyReLU()(block1)

        # Construct further blocks dynamically using configurations:
        for f in self.opt['discriminator']['filter_size']:
            # Apply consecutive discriminator convolution blocks with increasing complexity & depth,
            # as specified by the configuration. Each block processes the output of the previous one.
            block1 = discriminator_conv_block(f, block1)

        # Add a minibatch standard deviation layer to introduce more variability into the discriminator's inputs,
        # helping to avoid mode collapse by giving the discriminator a way to detect batch-level patterns.
        block1 = MinibatchStandardDeviation()(block1)
        # Apply another convolution with the final specified filter size to further process features before final decision.
        block1 = ScaledConv2d(channel_count=self.opt['discriminator']['filter_size'][-1][0],
                            kernel_size=3, strides=1, padding='same')(block1)
        block1 = ScaledLeakyReLU()(block1)
        # Final convolution that collapses all spatial dimensions, effectively producing a single feature per example.
        block1 = ScaledConv2d(channel_count=self.opt['discriminator']['filter_size'][-1][0],
                            kernel_size=int(block1.shape[1]), strides=1, padding='valid')(block1)
        block1 = ScaledLeakyReLU()(block1)
        # Flatten the output to prepare it for the final classification layer.
        block1 = tf.keras.layers.Flatten()(block1)
        # The final dense layer that outputs a single scalar value, representing the discriminator's confidence
        # in whether the input is real or fake.
        outputs = ScaledDense(1)(block1)

        # Construct & return the keras model:
        # The model takes the generator output (or real images) as input & provides a single scalar value as output.
        return tf.keras.Model(inputs=inputs, outputs=outputs)

    def reduce_across_batch(self, x: tf.Tensor) -> tf.Tensor:
        """
        Reduces the input tensor across the batch by computing the mean.

        Args:
            x (tf.Tensor): Input tensor to be reduced.

        Returns:
            tf.Tensor: Scalar tensor obtained by reducing `x`.
        """
        return tf.reduce_sum(x) / self.opt['batch_size']

    def parse_function(self, example_proto: tf.Tensor) -> tf.Tensor:
        """
        Parse & process data from a TFRecord file.

        Args:
            example_proto (tf.Tensor): A tensor containing a serialized TFRecord example.

        Returns:
            tf.Tensor: A tensor containing a processed image.
        """
        # Define the features in the TFRecord that are to be extracted.
        feature_description = {
            'image_raw': tf.io.FixedLenFeature([], tf.string),
        }
        # Parse the input `tf.train.Example` proto using the dictionary above.
        example = tf.io.parse_single_example(example_proto, feature_description)
        # Decode the image, assume RGB channels.
        image = tf.io.decode_png(example['image_raw'], channels=3)
        # Convert the image to floating point values & normalize the image to the range [0, 1].
        image = tf.cast(image, tf.float32) / 255.0

        return image

    def load_dataset(self) -> tf.data.Dataset:
        """
        Load & parse the dataset from specified TFRecord files.

        Returns:
            tf.data.Dataset: A TensorFlow Dataset object containing processed images.
        """
        # Get all available tfrecords
        tfrecord_paths = tf.io.gfile.glob(os.path.join(self.opt['dir_tfrecords'], '*.tfrecord'))

        # Create a dataset from the file paths.
        dataset = tf.data.TFRecordDataset(tfrecord_paths)
        # Map parsing function across dataset elements with parallel processing.
        dataset = dataset.map(self.parse_function, num_parallel_calls=tf.data.experimental.AUTOTUNE)

        # # Shuffle the dataset using the buffer size specified in the configuration.
        # dataset = dataset.shuffle(self.opt['buffer_size'])

        # Batch the dataset with the specified batch size from the configuration.
        dataset = dataset.batch(self.opt['batch_size'])

        # Take the specified number of batches to ensure each epoch has a consistent number of batches.
        dataset = dataset.take(self.opt['batch_num'])

        # Prefetch the dataset to improve training efficiency.
        dataset = dataset.prefetch(tf.data.experimental.AUTOTUNE)

        # Return the fully prepared dataset.
        return dataset

    def latent(self) -> tf.Tensor:
        """
        Generates a batch of latent vectors.

        Returns:
            tf.Tensor: A batch of random latent vectors shaped according to the generator's input requirements.
        """
        return tf.random.normal((self.opt['batch_size'], self.opt['generator']['latent_size']))

    @tf.function # What are decorators? See https://www.datacamp.com/tutorial/decorators-python
    def generator_step(self) -> tf.Tensor:
        """
        Executes one generator training step within the GAN.

        This method encapsulates the process of a single training iteration for the generator component of a GAN.
        It uses TensorFlow's automatic differentiation capabilities to update the generator's weights based on the computed loss.
        The `@tf.function` decorator is used to convert this method into a TensorFlow graph operation, enhancing performance.

        Returns:
            tf.Tensor: The computed loss for the generator in this step. This value is used to monitor training progress & optimize the generator.
        """
        # Generate a batch of noise vectors using a method defined in this class. This noise serves as the input to the generator.
        noise = self.latent()

        # Pass the noise through the generator to produce fake images. The `training=True` flag enables
        # behaviors specific to training like batch normalization & dropout within the generator.
        fake_images = self.generator(noise, training=True)

        # Pass the fake images through the discriminator. Here, `training=False` is set to ensure that the discriminator's
        # batch normalization layers (if any) are in inference mode, not affecting its learned statistics.
        fake_classifications = self.discriminator(fake_images, training=False)

        # Calculate the generator's loss using the softplus function & the negative of discriminator classifications.
        # This loss encourages the generator to produce images that are classified as real by the discriminator.
        loss = self.reduce_across_batch(tf.nn.softplus(-fake_classifications))

        # Compute the gradients of the loss with respect to the generator's trainable variables.
        grads = tf.gradients(loss, self.generator.trainable_variables)

        # Apply the calculated gradients to the generator's trainable variables using its optimizer.
        # This step updates the weights of the generator to improve its performance.
        self.generator.optimizer.apply_gradients(zip(grads, self.generator.trainable_variables))

        # Return the loss computed in this step, providing feedback on how well the generator is performing.
        return loss

    @tf.function # What are decorators? See https://www.datacamp.com/tutorial/decorators-python
    def discriminator_step(self, real_images: tf.Tensor) -> Tuple[tf.Tensor, tf.Tensor, tf.Tensor]:
        """
        Executes one discriminator training step using both real & generated images.

        This method manages the training process for the discriminator part of a GAN. It involves processing both real & fake images,
        calculating losses associated with each, & then updating the discriminator's weights. The `@tf.function` decorator converts
        this method into a highly efficient TensorFlow graph operation, significantly enhancing execution performance.

        Args:
            real_images (tf.Tensor): A batch of real images that the discriminator will try to classify as real.

        Returns:
            Tuple[tf.Tensor, tf.Tensor, tf.Tensor]: Tuple containing the total discriminator loss,
                                                    real loss (loss computed from real images), and
                                                    fake loss (loss computed from fake images).
        """
        # Generate a batch of noise vectors which will be used to generate fake images.
        noise = self.latent()

        # Generate fake images from the noise, with `training=False` to keep the generator in inference mode,
        # ensuring it doesn't update any internal statistics or apply dropout, etc.
        fake_images = self.generator(noise, training=False)

        # Classify the batch of real images using the discriminator. `training=True` indicates
        # that the discriminator should update its internal statistics during this forward pass.
        real_classifications = self.discriminator(real_images, training=True)

        # Classify the batch of fake images using the discriminator. As with real images,
        # `training=True` is used so the discriminator updates its internal statistics based on fake images as well.
        fake_classifications = self.discriminator(fake_images, training=True)

        # Calculate the loss for real images using softplus, designed to penalize the discriminator
        # when it fails to classify real images as real.
        real_loss = self.reduce_across_batch(tf.nn.softplus(-real_classifications))

        # Calculate the loss for fake images using softplus, designed to penalize the discriminator
        # when it fails to classify fake images as fake.
        fake_loss = self.reduce_across_batch(tf.nn.softplus(fake_classifications))

        # Calculate the total discriminator loss as the sum of real & fake losses.
        d_loss = real_loss + fake_loss

        # Compute gradients of the total loss with respect to the discriminator’s trainable variables.
        d_grads = tf.gradients(d_loss, self.discriminator.trainable_variables)

        # Apply the computed gradients to the discriminator's variables using its optimizer.
        # This updates the discriminator to improve its classification accuracy.
        self.discriminator.optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_variables))

        # Return the total loss, real loss, & fake loss, providing insight into how well the discriminator is learning
        # to distinguish between real & fake images.
        return d_loss, real_loss, fake_loss

    @tf.function # What are decorators? See https://www.datacamp.com/tutorial/decorators-python
    def regularization_step(self, real_images: tf.Tensor) -> tf.Tensor:
        """
        Applies regularization to the discriminator based on the gradients of real images.

        This method is designed to implement gradient penalty regularization, which helps in stabilizing the training of the discriminator
        by penalizing the gradient norms of the discriminator's output with respect to real images. This approach is crucial for enforcing
        the Lipschitz constraint & improving the convergence properties of the discriminator.

        Args:
            real_images (tf.Tensor): A batch of real images used for gradient penalty computation. These images are processed by the
                                    discriminator to compute gradients which are then used to calculate the penalty.

        Returns:
            tf.Tensor: Computed gradient penalty, a scalar tensor representing the penalty to be applied to the discriminator's loss.
        """
        # Classify real images using the discriminator; `training=True` enables updates to batch normalization layers based on these images.
        real_classifications = self.discriminator(real_images, training=True)

        # Compute gradients of the discriminator's classifications with respect to the input images.
        # This gradient reflects how changes in input images affect the discriminator's decision-making process.
        real_grads = tf.gradients(tf.reduce_sum(real_classifications), real_images)

        # Compute the L2 norm of the gradients for each image, then average across the batch.
        # Squaring & summing the components of gradients for each image helps measure how "steep" these gradients are.
        gradient_loss = self.reduce_across_batch(tf.reduce_sum(tf.square(real_grads), axis=[1, 2, 3]))

        # Calculate the gradient penalty strength, scaling it by a factor from the configuration
        # & typically involves a hyperparameter to balance the regularization strength.
        gradient_penalty_strength = 10. * 0.5 * self.opt['discriminator']['regularization_interval']

        # Multiply the average gradient norm by the penalty strength to compute the total gradient penalty.
        gradient_penalty = gradient_loss * gradient_penalty_strength

        # Compute gradients of the gradient penalty with respect to the discriminator's trainable variables.
        reg_grads = tf.gradients(gradient_penalty, self.discriminator.trainable_variables)

        # Set the last gradients to zero. This can be a specific technique to avoid regularizing certain components
        # or due to specific architectural needs.
        reg_grads[-1] = tf.zeros_like(self.discriminator.trainable_variables[-1])

        # Apply these computed gradients to the discriminator's variables to update them in a way that minimizes the gradient penalty.
        self.discriminator.optimizer.apply_gradients(zip(reg_grads, self.discriminator.trainable_variables))

        # Return the computed gradient penalty as a scalar tensor, which quantifies the enforced smoothness of the discriminator's function.
        return gradient_penalty

    def plot(self):
        """
        Generates & saves a grid of generated images to monitor the progress of the generator.

        This method facilitates the visual inspection of the generator's output by creating & saving a grid of images
        that the generator produces from random noise inputs. This visual output can be crucial for understanding the generator's
        learning progress & adjusting training parameters as needed.

        """
        # Generate a batch of noise vectors using the latent function.
        # This noise will be transformed by the generator into synthetic images.
        noise = self.latent()

        # Generate fake images using the generator with the noise as input.
        # The `training=False` parameter ensures the generator uses inference mode settings,
        # which is important for consistent image output.
        fake_images = self.generator(noise, training=False)

        # Determine the number of rows & columns for the image grid.
        # The grid size is set based on the batch size to accommodate all generated images.
        num_rows = int(max(1, np.sqrt(self.opt['batch_size'])))
        num_columns = int(max(1, np.sqrt(self.opt['batch_size'])))

        # Initialize variables for constructing the grid.
        counter = 0
        image = []

        # Iterate over each row to assemble individual image tensors into a row tensor.
        for row in range(num_rows):
            image_row = []
            # Iterate over each column to append each image tensor to the row tensor list.
            for column in range(num_columns):
                # Append the current image to the row list & increment the counter to move to the next image.
                image_row.append(fake_images[counter, :, :, :])
                counter += 1
            # Concatenate all images in a row horizontally to form a single row tensor.
            image.append(tf.concat(image_row, axis=1))

        # Concatenate all row tensors vertically to form the final grid tensor.
        image = tf.concat(image, axis=0)

        # Define the filename & path where the grid image will be saved.
        # This path is specified in the configuration options under 'dir_samples'.
        filename = os.path.join(self.opt['dir_samples'], f'samples_{int(time.time())}.png')

        # remove samples when having at least 10 samples in the directory
        previous_images = glob.glob(os.path.join(self.opt['dir_samples'],'*.png'))
        if len(previous_images) >= 10:
            # Loop through the list of file paths & remove each file
            for file_path in previous_images:
                try:
                    os.remove(file_path)
                except Exception as e:
                    print(f"Error removing {file_path}: {e}")

        # Save the final image grid to the specified filename.
        # This function handles converting the tensor to a valid image format.
        tf.keras.utils.save_img(filename, image)

    def model_save(self):
        """
        Saves the weights of the generator & discriminator models.

        This method provides functionality to persist the current state of the generator & discriminator models by saving their
        weights. This is crucial for checkpointing the models during training, allowing for recovery & continuation of training
        at a later time, or using the trained models for inference without needing to retrain.

        """
        # Define the filename & path for saving the generator's weights.
        # The directory for storing checkpoints is specified in the configuration options under 'dir_checkpoints'.
        filename = os.path.join(self.opt['dir_checkpoints'], 'generator.h5')
        # Save the weights of the generator model to the specified file.
        # The .h5 file format is used here, which is a common storage format for model weights in Keras.
        self.generator.save_weights(filename)

        # Define the filename & path for saving the discriminator's weights.
        filename = os.path.join(self.opt['dir_checkpoints'], 'discriminator.h5')
        # Save the weights of the discriminator model to the specified file.
        # Saving the discriminator's weights is equally important as it allows for the entire GAN architecture to be restored.
        self.discriminator.save_weights(filename)

    def training(self):
        """
        Conducts the training process for the GAN, including both generator & discriminator components.

        This method manages the entire training lifecycle of a GAN. It sets up optimizers for both models,
        loads any existing model weights, & iteratively trains the generator & discriminator through a series
        of epochs. It includes checkpoints & plotting intervals for monitoring & saving progress, & applies
        regularization to ensure the stability of the training process.

        """
        # Set up the Adam optimizer for the generator with specified learning rate & beta coefficients.
        # These parameters are pulled from the configuration dictionary (`self.opt`).
        self.generator.optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.opt['generator']['lr'],
            beta_1=self.opt['generator']['beta1'],
            beta_2=self.opt['generator']['beta2']
        )

        # Set up the Adam optimizer for the discriminator.
        # The learning rate & beta coefficients are scaled by a 'lazy_ratio' from the configuration,
        # which is used to adjust the training speed relative to the generator.
        self.discriminator.optimizer = tf.keras.optimizers.Adam(
            learning_rate=self.opt['discriminator']['lazy_ratio'] * self.opt['generator']['lr'],
            beta_1=self.opt['discriminator']['lazy_ratio'] * self.opt['generator']['beta1'],
            beta_2=self.opt['discriminator']['lazy_ratio'] * self.opt['generator']['beta2']
        )

        # Load previously saved model weights if available, enabling continuation of training from a checkpoint.
        self.model_load()

        # Create a dataset. This dataset includes preprocessing & batching of images.
        dataset = self.load_dataset()

        # Begin training over the specified number of epochs.
        for epoch in range(self.opt['epochs']):

            # Initiate batch numbering
            batch   = 0
            # Initiate running loss for both generator & discriminator
            running_d_loss = 0
            running_g_loss = 0

            # Initiate a progress bar using tqdm library
            pbar = tqdm(dataset, total=self.opt['batch_num'], desc=f"Epoch: {epoch+1:04d}", ncols=150,leave = False)

            for real_images in pbar:

                # Control if we do not encounter & incomplete final batch
                if len(real_images) == self.opt['batch_size']:
                    # Perform a training step for the discriminator using the real images, & capture the loss.
                    d_loss, _, _ = self.discriminator_step(real_images=real_images)

                    # Regularly apply gradient penalty as a regularization method to stabilize the discriminator,
                    # based on the specified interval in the configuration.
                    if (batch + 1) % self.opt['discriminator']['regularization_interval'] == 0:
                        _ = self.regularization_step(real_images=real_images)

                    # Perform a training step for the generator & capture the resulting loss.
                    g_loss = self.generator_step()

                    # At specified intervals, generate & save a grid of images to visually monitor the generator's progress.
                    if (batch + 1) % self.opt['plot_intervals'] == 0:
                        self.plot()

                    # Save the model at specified intervals, creating checkpoints for long-running training processes.
                    if (batch + 1) % self.opt['checkpoint_intervals'] == 0:
                        self.model_save()

                    running_d_loss += d_loss
                    running_g_loss += g_loss


                    # Add running loss information to the progress bar.
                    pbar.set_postfix(loss=f"Discriminator Loss: {running_d_loss/(batch+1):10.8f} | Generator Loss: {running_g_loss/(batch+1):10.8f}")

                # update the batch
                batch += 1

    def model_load(self):
        """
        Loads the weights of the generator & discriminator models if available.

        This method checks for the presence of saved weight files for both the generator & discriminator models & loads them
        if they exist. Loading model weights is essential for resuming training from a checkpoint, deploying models for inference,
        or continuing experiments without needing to retrain models from scratch.

        """
        try:
            # Define the filename & path for the generator's saved weights.
            # The directory where checkpoints are stored is specified in the configuration options under 'dir_checkpoints'.
            filename = os.path.join(self.opt['dir_checkpoints'], 'generator.h5')
            # Check if the file with the saved weights exists.
            if os.path.exists(filename):
                # Load the weights into the generator model if the file exists.
                # This restores the generator's state to the last checkpointed version, enabling continuity in training or deployment.
                self.generator.load_weights(filename)

            # Define the filename & path for the discriminator's saved weights.
            filename = os.path.join(self.opt['dir_checkpoints'], 'discriminator.h5')
            # Check if the file with the saved weights exists.
            if os.path.exists(filename):
                # Load the weights into the discriminator model if the file exists.
                # Similarly to the generator, this restores the discriminator's state to the last checkpointed version.
                self.discriminator.load_weights(filename)
        except:
            pass

    def generate_new_images(self):
        """
        Runs inference with the generator to produce & save images.

        This method loads the trained generator & discriminator models & then uses the generator to create images from random
        noise inputs. Each generated image is evaluated by the discriminator, & its score is used to name the image file, reflecting
        the discriminator's confidence in the image being realistic. The images are saved for later viewing or analysis.

        """
        # Load the latest weights for the generator & discriminator models to ensure the inference uses the trained models.
        self.model_load()

        # Run the inference process for a specified number of iterations, defined in the model's configuration options.
        for i in range(self.opt['inference_num']):
            # Generate a random noise vector with the specified latent size. This vector acts as the input to the generator.
            noise = tf.random.normal((1, self.opt['generator']['latent_size']))

            # Generate an image by passing the noise through the generator.
            # `training=False` ensures the generator is in inference mode, using learned statistics rather than updating them.
            image = self.generator(noise, training=False)

            # Evaluate the generated image using the discriminator to get a 'realism' score.
            # `training=False` here ensures the discriminator is also in inference mode.
            score = self.discriminator(image, training=False)

            # Create a filename for saving the image. The name includes the discriminator's score as a large integer for uniqueness.
            name = f"{int(float(score) * 1e19):020d}.png"

            # Construct the full path where the image will be saved. This directory is specified in the model's configuration options.
            filename = os.path.join(self.opt['dir_inferences'], name)

            # Save the generated image to the specified path. Only the image tensor is saved, stripping any batch dimensions.
            tf.keras.utils.save_img(filename, image[0, :, :, :])


In [ ]:
#@title Main Run - Training
# Let's connect to Google Drive to manage our files efficiently.
# The free version of Google Drive gives us 15GB of disk space as of May 2024.
# Follow the instructions to give access to Google Drive & then comeback to this tab.
from google.colab import drive
drive.mount('/content/drive')

# Import necessary libraries
import os

# Get the input arguments
args = Args()

import glob
import copy
import math
import random
import time
import shutil
import requests


import tensorflow as tf
import numpy as np
import pandas as pd

from tqdm import tqdm
from tensorflow.python.keras.utils import conv_utils
from matplotlib import pyplot as plt
from typing import *

# Suppress TensorFlow warnings
# Here's how you can set TF_CPP_MIN_LOG_LEVEL to different levels:
# 0: Default, shows all logs (DEBUG, INFO, WARN, ERROR, & FATAL).
# 1: Filters out INFO logs.
# 2: Additionally filters out WARNING logs.
# 3: Additionally filters out ERROR logs, showing only FATAL.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Execute the operational logic of the script initializing data, models, and
# performing training or inference based on the mode.

# Initialize with options
opt    = initiate(args)

# Download the data.
# You can use the records of any image data repository with minor changes to the code.
# Here, we have the records of 32 by 32-pixel animal faces.
# Source: https://www.kaggle.com/datasets/dimensi0n/afhq-512.
# Let's download the data using the' "wget" Linux command.
# First, make sure that the data is not already in the Google Drive designated folder.
tfrecords_path = glob.glob(os.path.join(opt['dir_tfrecords'], '*'))
if len(tfrecords_path) == 0:
    !wget https://storage.googleapis.com/535743/gan/animal_faces/animal_faces_128.zip
    # Unzip the files into the tfrecords folder.
    !unzip animal_faces_128.zip -d {opt['dir_tfrecords']}
    # remove the zip file
    !rm animal_faces_128.zip
    # Delay for Google Drive usual time lags.
    # If the cell runs without any training, you need to run this cell in a few minutes (15 minutes)
    # The reason is sometimes it takes time for Colab to be fully synched with Google Drive when
    # we copy files over there. For larger files (image_size), you need to increase this delay.
    time.sleep(900)

    # reboot the instance
    !sudo reboot

# Create the GAN object.
my_obj = MyGenerativeAdversarialModel(opt)

# Print generator model summary.
my_obj.generator.summary()

time.sleep(5) # Delay for readability of output.

print("\n$$$$$$$$$$$$$$$$$$$$$$$$$$\n")

# Print discriminator model summary.
my_obj.discriminator.summary()
time.sleep(5) # Delay for readability of output.


if args.gpu:
    gpus = tf.config.list_physical_devices('GPU')
    with tf.device(gpus[0].name.replace('/physical_','')):
        # Train with one (first) GPU.
        if args.inference_mode:
            my_obj.generate_new_images()
        else:
            my_obj.training()
else:
    cpus = tf.config.list_physical_devices('CPU')
    with tf.device(cpus[0].name.replace('/physical_','')):
        # Train with one (first) GPU
        if args.inference_mode:
            my_obj.generate_new_images()
        else:
            my_obj.training()


### **5.4. Inference**

In the inference, we can now use our generator to generate fake images of animals. The codes are similar to what we have seen before, with only minor changes in the Args class.

In [ ]:
#@title Class - Args for Inference
'''
No required library.
'''

class Args():
    def __init__(self):
        """
        Initializes the configuration settings for a GAN training session.

        This class sets up various configuration parameters that control the training process, model architecture,
        & operational settings such as data handling & output management. These settings are crucial for customizing
        the behavior of the GAN according to specific experimental needs.
        """

        '''
        General Mandatory
        '''

        # Name of the experiment, used for organizing outputs & logs.
        self.experiment_name = 'animal_faces'
        # Type of the project
        self.project_type = 'gan'
        # The project patent folder
        self.project_parent_folder = '/content/drive/MyDrive'

        '''
        General Optional
        '''

        # The number of samples per batch to be used during training.
        self.batch_size = 128
        # Total number of epochs to train the models.
        self.epochs = 500
        # # Number of batches to process during the training. This can control how long an "epoch" is if dataset is very large.
        # self.batch_num = 60
        # Buffer size for shuffling the dataset. Larger sizes can improve shuffling quality but use more memory.
        self.buffer_size = 1000
        # Interval in batches to plot the generator output during training, helps monitor progress visually.
        self.plot_intervals = 50
        # Interval in batches to save checkpoints of the model weights.
        self.checkpoint_intervals = 50
        # Target size of images (height & width) that the generator should output.
        self.image_size = 128 # supported sizes are 2^n where n = [5,6,...,13]
        # If true, removes existing data from the folders like checkpoints & samples before starting new training.
        self.folder_remove = False
        # If true, runs the model in inference mode to generate images without training.
        self.inference_mode = True
        # Number of images to generate if running in inference mode.
        self.inference_num = 25
        # If true, forces the training or inference to run on CPU instead of GPU.
        self.gpu = False


        '''
        Generator
        '''

        # Size of the latent vector used as input to the generator.
        self.generator_latent_size = 512
        # Factor by which to reduce the filter size of each Conv layer from a baseline setting, affecting model capacity (2^n).
        self.generator_reduce_factor = 16 # acceptable values: 1, 2, 4, 8
        # Learning rate for the generator's optimizer.
        self.generator_lr = 0.002
        # First moment decay rate (beta1) for the Adam optimizer in the generator.
        self.generator_beta1 = 0.0
        # Second moment decay rate (beta2) for the Adam optimizer in the generator.
        self.generator_beta2 = 0.99

        '''
        Discriminator
        '''

        # Factor by which to reduce the filter size of each Conv layer in the discriminator, similar to generator's setting (2^n).
        self.discriminator_reduce_factor = 16 # acceptable values: 1, 2, 4, 8
        # Learning rate for the discriminator's optimizer.
        self.discriminator_lr = 0.002
        # First moment decay rate (beta1) for the Adam optimizer in the discriminator.
        self.discriminator_beta1 = 0.0
        # Second moment decay rate (beta2) for the Adam optimizer in the discriminator.
        self.discriminator_beta2 = 0.99
        # Interval at which to apply regularization techniques during discriminator training.
        self.discriminator_regularization_interval = 16


In [ ]:
#@title Main Run - Inference
# Let's connect to Google Drive to manage our files efficiently.
# The free version of Google Drive gives us 15GB of disk space as of May 2024.
# Follow the instructions to give access to Google Drive & then comeback to this tab.
from google.colab import drive
drive.mount('/content/drive')

# Import necessary libraries
import os

# Get the input arguments
args = Args()

import glob
import copy
import math
import random
import time
import shutil
import requests


import tensorflow as tf
import numpy as np
import pandas as pd

from tqdm import tqdm
from tensorflow.python.keras.utils import conv_utils
from matplotlib import pyplot as plt
from typing import *

# Suppress TensorFlow warnings
# Here's how you can set TF_CPP_MIN_LOG_LEVEL to different levels:
# 0: Default, shows all logs (DEBUG, INFO, WARN, ERROR, & FATAL).
# 1: Filters out INFO logs.
# 2: Additionally filters out WARNING logs.
# 3: Additionally filters out ERROR logs, showing only FATAL.
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# Execute the operational logic of the script initializing data, models, and
# performing training or inference based on the mode.

# Initialize with options
opt    = initiate(args)

# Create the GAN object.
my_obj = MyGenerativeAdversarialModel(opt)

# Print generator model summary.
my_obj.generator.summary()

time.sleep(5) # Delay for readability of output.

print("\n$$$$$$$$$$$$$$$$$$$$$$$$$$\n")

# Print discriminator model summary.
my_obj.discriminator.summary()
time.sleep(5) # Delay for readability of output.


if args.gpu:
    gpus = tf.config.list_physical_devices('GPU')
    with tf.device(gpus[0].name.replace('/physical_','')):
        # Train with one (first) GPU.
        if args.inference_mode:
            my_obj.generate_new_images()
        else:
            my_obj.training()
else:
    cpus = tf.config.list_physical_devices('CPU')
    with tf.device(cpus[0].name.replace('/physical_','')):
        # Train with one (first) GPU
        if args.inference_mode:
            my_obj.generate_new_images()
        else:
            my_obj.training()


# <font color="#418FDE" size="6.5" uppercase>**B: StyleGAN Experiment**</font>
----

In this lecture, you learned to:
* Develop [Style](https://arxiv.org/pdf/1812.04948.pdf)/[Pro](https://arxiv.org/pdf/1710.10196.pdf)Gan-Like generative models in the TensorFlow ecosystem.
* Build Generative Adversarial Networks' (GANs) functions & classes.

In the next lecture (lecture C), we will develop a diffusion-like experiment.